# Latent Studio — SD 1.5 Image Generator

Creative Coding Advanced capstone. Generated from `src/latent_studio/` (see `scripts/build_colab_notebook.py`) — flattened into one notebook namespace, no separate files or imports between sections, so you can read and run it top-to-bottom like a single script.

The six shipped project LoRAs (Hokusai, Turner, Monet, Dürer, Hiroshige, Rembrandt) are produced by a **separate** notebook, `lora_training_colab.ipynb`, and loaded here from the Hugging Face Hub via the registry.

**Runtime:** switch to a GPU runtime (T4) before running the generation cells. Building/debugging the interface can be done on a CPU runtime.

## 1. Install dependencies

In [ ]:
#@title Install dependencies
%pip install -q "torch>=2.2" "torchvision>=0.17" "diffusers>=0.27" "transformers>=4.41" "accelerate>=0.31" "peft>=0.11" "safetensors>=0.4" "gradio>=6.0" "pillow>=10.0" "opencv-python-headless>=4.9" "numpy>=1.26" "huggingface_hub>=0.34"
%pip uninstall -q -y torchao

## 2. Hugging Face setup
Everything this app loads is public, so a token is not needed for *access* — but anonymous Hub requests are **rate-limited**, and this notebook pulls several GB of checkpoints. A plain **read** token removes the throttle; without it the first model load crawls.

The cell also **switches off Xet**, `huggingface_hub`'s default download backend, which hangs part-way into a large file on Colab (see the note in the source). Both settings are read when `huggingface_hub` is first imported, so **this cell must run before the imports below** — setting them later does nothing. If you have already imported anything, restart the runtime.

On Colab, store the token once as a secret named **`HF_TOKEN`** (🔑 icon in the left sidebar, 'Notebook access' on). Otherwise this falls back to an interactive login.

In [ ]:
#@title Hugging Face login
import os

# Read by huggingface_hub at import, so this must run before anything imports it.
# Xet is the default download backend and it hangs in Colab (see the note above).
os.environ["HF_HUB_DISABLE_XET"] = "1"


def _hf_token():
    if os.environ.get("HF_TOKEN"):
        return "HF_TOKEN env var"
    try:  # Colab secret (key icon in the sidebar, 'Notebook access' ON)
        from google.colab import userdata

        token = userdata.get("HF_TOKEN")
        if token:
            os.environ["HF_TOKEN"] = token
            return "Colab secret"
    except Exception:
        pass  # not on Colab, or no secret set / not shared with this notebook
    from huggingface_hub import get_token

    return "cached login" if get_token() else None


source = _hf_token()
if source is None:
    print(
        "No token found — anonymous downloads are rate-limited, so logging in.\n"
        "Tip: save it once as a Colab secret named HF_TOKEN (key icon, left sidebar)\n"
        "and this cell will pick it up silently from now on."
    )
    from huggingface_hub import login

    login()
else:
    print(f"Hugging Face token found ({source}) — downloads are authenticated and fast.")

## 3. Imports and Globals

In [ ]:
#@title Imports
import torch
import gc
import os
import shutil
import cv2
import numpy as np
import json
import threading
import time
import gradio as gr
import html
import math
import re
import tempfile
import random
from dataclasses import dataclass, asdict, field, replace
from typing import Callable
from diffusers import StableDiffusionPipeline, ControlNetModel, StableDiffusionControlNetPipeline, UniPCMultistepScheduler
from huggingface_hub import hf_hub_download
from PIL import Image, ImageDraw, ImageFont
from PIL.PngImagePlugin import PngInfo
from urllib.parse import quote
from contextlib import contextmanager
from tqdm.std import tqdm as _std_tqdm


## 4. App modules (reference)
One card per file in `src/latent_studio/`, in the order they're defined — run them all to build the app below. Skip ahead if you're only here to launch it.

### Device detection
_(`src/latent_studio/device.py`)_

Picks `cuda` / `mps` / `cpu` and the matching dtype (fp16 everywhere except plain CPU) so the exact same code runs on this Colab GPU and on a MacBook's Apple Silicon (MPS) during local development. Nothing here needs tweaking.

In [ ]:
#@title device.py
"""Device/dtype selection — same cuda/mps/cpu logic as pipeline_V5, so the app
behaves identically on the local MacBook (MPS) and on Colab (CUDA T4)."""

def get_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"

def get_dtype() -> torch.dtype:
    # fp16 everywhere except plain CPU (brief requires fp16 for VRAM budget on the T4).
    if torch.cuda.is_available() or torch.backends.mps.is_available():
        return torch.float16
    return torch.float32

def empty_cache(device: str) -> None:
    if device == "cuda":
        torch.cuda.empty_cache()
        if hasattr(torch.cuda, "ipc_collect"):
            torch.cuda.ipc_collect()
    elif device == "mps" and hasattr(torch.mps, "empty_cache"):
        torch.mps.empty_cache()


### Model & LoRA registry
_(`src/latent_studio/registry.py`)_

The single source of truth for what shows up in the app's Model and Style pickers. **Tweak `CHECKPOINTS`** to add/remove curated SD1.5 checkpoints (any Hugging Face repo compatible with `StableDiffusionPipeline`). **`LORAS`** points at the six shipped project LoRAs (Hokusai, Turner, Monet, Dürer, Hiroshige, Rembrandt) by their Hugging Face repo id — those are produced by the separate LoRA training notebook (`lora_training_colab.ipynb`) and download from the Hub automatically. Selecting a LoRA before it has been trained/pushed just fails gracefully.

In [ ]:
#@title registry.py
"""Curated checkpoint/LoRA registry — single source of truth for the Gradio
selectors. Keep display names short (they go directly into the UI labels)."""

@dataclass(frozen=True)
class Checkpoint:
    id: str
    label: str
    repo_id: str
    license: str
    note: str = ""

@dataclass(frozen=True)
class LoRA:
    id: str
    label: str
    path: str | None  # local path or HF repo id; None for the "no LoRA" entry
    trigger_word: str = ""
    license: str = ""
    note: str = ""

CHECKPOINTS: list[Checkpoint] = [
    Checkpoint(
        id="sd15-base",
        label="Stable Diffusion 1.5",
        repo_id="stable-diffusion-v1-5/stable-diffusion-v1-5",
        license="CreativeML OpenRAIL-M",
    ),
    Checkpoint(
        id="dreamshaper8",
        label="DreamShaper 8",
        repo_id="Lykon/dreamshaper-8",
        license="CreativeML OpenRAIL-M",
        note="Community SD1.5 checkpoint, stylized/photoreal blend.",
    ),
    Checkpoint(
        id="epicrealism",
        label="epiCRealism",
        repo_id="emilianJR/epiCRealism",
        license="CreativeML OpenRAIL-M",
        note="Community SD1.5 checkpoint, photoreal focus.",
    ),
]

DEFAULT_CHECKPOINT_ID = CHECKPOINTS[0].id

# The project LoRAs, produced by training/ and published to the Hugging Face Hub —
# `path` is the HF repo id, downloaded on first use. Repo ids + trigger words must
# match training/config.py. This is the SHIPPED roster (six of the nine trained
# LoRAs; Cézanne/Cassatt/Van Gogh stay documented limitations, not shipped).
LORAS: list[LoRA] = [
    LoRA(id="none", label="No style", path=None),
    LoRA(
        id="hokusai",
        label="Katsushika Hokusai",
        path="espressosession/latent-studio-hokusai-lora",
        trigger_word="sks",  # shared neutral trigger — matches config.TRIGGER_WORD
        license="CreativeML OpenRAIL-M (LoRA); training data CC0 (Art Institute of Chicago)",
        note="Style LoRA — Katsushika Hokusai woodblock prints, public domain.",
    ),
    LoRA(
        id="turner",
        label="J.M.W. Turner",
        path="espressosession/latent-studio-turner-lora",
        trigger_word="sks",  # shared neutral trigger — matches config.TRIGGER_WORD
        license="CreativeML OpenRAIL-M (LoRA); training data CC0 (Art Institute of Chicago)",
        note="Style LoRA — J.M.W. Turner oil landscapes/seascapes, public domain.",
    ),
    LoRA(
        id="monet",
        label="Claude Monet",
        path="espressosession/latent-studio-monet-lora",
        trigger_word="sks",
        license="CreativeML OpenRAIL-M (LoRA); training data CC0 (Art Institute of Chicago)",
        note="Style LoRA — Claude Monet Impressionist landscapes, public domain.",
    ),
    LoRA(
        id="duerer",
        label="Albrecht Dürer",
        path="espressosession/latent-studio-duerer-lora",
        trigger_word="sks",
        license="CreativeML OpenRAIL-M (LoRA); training data CC0 (Art Institute of Chicago)",
        note="Style LoRA — Albrecht Dürer Renaissance engravings/woodcuts, public domain.",
    ),
    LoRA(
        id="hiroshige",
        label="Utagawa Hiroshige",
        path="espressosession/latent-studio-hiroshige-lora",
        trigger_word="sks",
        license="CreativeML OpenRAIL-M (LoRA); training data CC0 (Art Institute of Chicago)",
        note="Style LoRA — Utagawa Hiroshige ukiyo-e landscapes, public domain.",
    ),
    LoRA(
        id="rembrandt",
        label="Rembrandt van Rijn",
        path="espressosession/latent-studio-rembrandt-lora",
        trigger_word="sks",
        license="CreativeML OpenRAIL-M (LoRA); training data CC0 (Art Institute of Chicago)",
        note="Style LoRA — Rembrandt etchings, chiaroscuro, public domain.",
    ),
]

DEFAULT_LORA_ID = "none"

def get_checkpoint(checkpoint_id: str) -> Checkpoint:
    for cp in CHECKPOINTS:
        if cp.id == checkpoint_id:
            return cp
    raise KeyError(f"Unknown checkpoint id: {checkpoint_id}")

def get_lora(lora_id: str) -> LoRA:
    for lora in LORAS:
        if lora.id == lora_id:
            return lora
    raise KeyError(f"Unknown LoRA id: {lora_id}")


### Pipeline manager
_(`src/latent_studio/pipeline_manager.py`)_

Loads/unloads the SD1.5 base checkpoint and LoRA weights, keeping exactly one active pipeline at a time. Switching the **base model** is the expensive operation — it fully reloads and frees the old pipeline's memory first. Switching **which LoRA** is loaded is cheaper; changing only the LoRA **weight** (strength) is free — it's applied per-generation instead of reloading anything. **Tweak:** `DISABLE_SAFETY_CHECKER` is a local-only debug escape hatch — set it before importing this module to skip the safety checker while iterating offline. This notebook never sets it: the deployed app always runs with the safety checker enabled.

In [ ]:
#@title pipeline_manager.py
"""Loads/unloads the SD1.5 base checkpoint and LoRA weights, keeping exactly one
active pipeline at a time. Base-model switches fully reload; LoRA identity
switches are cheap; LoRA *weight* is free — applied per-call, not reloaded."""

StatusCallback = Callable[[str], None] | None

# Dev-only local escape hatch, never wired into the UI. The deployed app always
# runs with the safety checker enabled — a brief requirement for the public app.
_SAFETY_CHECKER_DISABLED = bool(os.environ.get("DISABLE_SAFETY_CHECKER"))

# Where training/export.py stages locally-trained LoRAs; preferred over the HF
# repo id when present, so a fully-local training run drives the app with no network.
APP_LORAS_DIR = "app_loras"

def _noop(_: str) -> None:
    pass

def preload_models(on_status: StatusCallback = None) -> None:
    """Downloads every checkpoint + LoRA into the local cache without loading any
    into VRAM, so a live demo's first model switch isn't network-bound."""
    on_status = on_status or _noop
    for cp in CHECKPOINTS:
        on_status(f"Fetching {cp.label}…")
        try:
            # fp16 halves the download; not every checkpoint ships that variant.
            StableDiffusionPipeline.download(cp.repo_id, variant="fp16", use_safetensors=True)
        except Exception:
            StableDiffusionPipeline.download(cp.repo_id, use_safetensors=True)
    os.makedirs(APP_LORAS_DIR, exist_ok=True)
    for lora in LORAS:
        if lora.path is None:
            continue
        local = os.path.join(APP_LORAS_DIR, f"{lora.id}-lora.safetensors")
        if os.path.exists(local):
            continue  # a local training run already staged it
        on_status(f"Fetching {lora.label}…")
        weights = hf_hub_download(lora.path, "pytorch_lora_weights.safetensors")
        shutil.copy(weights, local)
    on_status("All models cached.")

class PipelineManager:
    def __init__(self) -> None:
        self.device = get_device()
        self.dtype = get_dtype()
        self.pipe: StableDiffusionPipeline | None = None
        self.checkpoint_id: str | None = None
        self.lora_id: str = "none"

    @property
    def is_ready(self) -> bool:
        return self.pipe is not None

    def load_checkpoint(self, checkpoint_id: str, on_status: StatusCallback = None) -> None:
        on_status = on_status or _noop
        if self.pipe is not None and self.checkpoint_id == checkpoint_id:
            return  # already loaded, nothing to do

        checkpoint = get_checkpoint(checkpoint_id)

        if self.pipe is not None:
            on_status(f"Unloading previous model ({self.checkpoint_id})...")
            self._unload_pipe()

        on_status(f"Loading {checkpoint.label}...")
        safety_kwargs = (
            {"safety_checker": None, "requires_safety_checker": False}
            if _SAFETY_CHECKER_DISABLED
            else {}
        )
        # low_cpu_mem_usage=False avoids accelerate's meta-device load path, which on
        # some torch/accelerate combos (notably Colab) leaves .to(device) unable to
        # copy ("Cannot copy out of meta tensor") — same fix as controlnet.py's load().
        kwargs = {
            "torch_dtype": self.dtype, "use_safetensors": True,
            "low_cpu_mem_usage": False, **safety_kwargs,
        }

        # Ask for fp16 weights directly rather than downloading fp32 and casting down;
        # not every checkpoint ships an fp16 variant (epiCRealism doesn't), so fall back.
        if self.dtype == torch.float16:
            try:
                pipe = StableDiffusionPipeline.from_pretrained(
                    checkpoint.repo_id, variant="fp16", **kwargs
                ).to(self.device)
            except Exception:
                on_status(f"No fp16 build for {checkpoint.label} — loading full weights...")
                pipe = StableDiffusionPipeline.from_pretrained(
                    checkpoint.repo_id, **kwargs
                ).to(self.device)
        else:
            pipe = StableDiffusionPipeline.from_pretrained(
                checkpoint.repo_id, **kwargs
            ).to(self.device)

        # is_ready reads `self.pipe is not None` and is polled from another thread (the
        # state panel) without a lock — checkpoint_id must already be correct before
        # self.pipe flips non-None, not after. Build into a local, assign checkpoint_id
        # first, self.pipe last.
        self.checkpoint_id = checkpoint_id
        self.lora_id = "none"
        self.pipe = pipe
        on_status(f"{checkpoint.label} ready.")

    def set_lora(self, lora_id: str, on_status: StatusCallback = None) -> None:
        on_status = on_status or _noop
        if self.pipe is None:
            raise RuntimeError("Load a base checkpoint before selecting a LoRA.")
        if lora_id == self.lora_id:
            return

        if self.lora_id != "none":
            self.pipe.unload_lora_weights()

        lora = get_lora(lora_id)
        if lora.path is not None:
            local = os.path.join(APP_LORAS_DIR, f"{lora.id}-lora.safetensors")
            source = local if os.path.exists(local) else lora.path
            on_status(f"Loading LoRA {lora.label}...")
            # low_cpu_mem_usage=False: diffusers defaults this to True whenever peft/
            # transformers are recent enough (true here and on Colab), which takes the
            # same accelerate meta-device path as the checkpoint load above — and here
            # it left a layer's bias in its original fp32 instead of the pipe's fp16
            # ("Input type (c10::Half) and bias type (float) should be the same").
            self.pipe.load_lora_weights(source, low_cpu_mem_usage=False)

        self.lora_id = lora_id
        on_status(f"LoRA set to {lora.label}.")

    def _unload_pipe(self) -> None:
        # Plain assignment, not `del self.pipe` — the state panel's polling loop reads
        # `is_ready` from a different thread, and `del` briefly leaves the attribute
        # missing entirely, which raced into an AttributeError in production.
        self.pipe = None
        self.checkpoint_id = None
        self.lora_id = "none"
        gc.collect()
        empty_cache(self.device)

    def unload(self) -> None:
        if self.pipe is not None:
            self._unload_pipe()

    def generator(self, seed: int) -> torch.Generator:
        gen_device = self.device if self.device != "mps" else "cpu"  # mps generator support is unreliable
        return torch.Generator(device=gen_device).manual_seed(seed)


### ControlNet (optional)
_(`src/latent_studio/controlnet.py`)_

Optional conditioning on an uploaded image via Canny edge detection and/or a Depth-Anything-V2 depth map, feeding into a `StableDiffusionControlNetPipeline` instead of the plain one. Supports one or both ControlNet types simultaneously (each with its own conditioning-scale weight). The rest of the app works fully without this section ever being used — it's defined here (before generation.py) only because generate() below type-hints against `ControlNetManager`.

In [ ]:
#@title controlnet.py
"""ADV (optional): ControlNet conditioning via Canny edges and/or
Depth-Anything-V2 depth maps, ported from L06/Controlnet_Canny_01_edited.ipynb.
Supports single or simultaneous multi-ControlNet with per-net conditioning
scales. The core app (app.py) works without ever importing this module."""

CONTROLNET_REPOS = {
    "canny": "lllyasviel/sd-controlnet-canny",
    "depth": "lllyasviel/sd-controlnet-depth",
}

_depth_processor = None
_depth_model = None

def canny_preprocess(image: Image.Image, low_threshold: int = 100, high_threshold: int = 200) -> Image.Image:
    array = np.array(image.convert("RGB"))
    edges = cv2.Canny(array, low_threshold, high_threshold)
    edges = np.stack([edges] * 3, axis=-1)
    return Image.fromarray(edges)

def _load_depth_model():
    global _depth_processor, _depth_model
    if _depth_model is None:
        from transformers import AutoImageProcessor, AutoModelForDepthEstimation

        _depth_processor = AutoImageProcessor.from_pretrained("Depth-Anything/Depth-Anything-V2-base-hf")
        _depth_model = AutoModelForDepthEstimation.from_pretrained("Depth-Anything/Depth-Anything-V2-base-hf")
    return _depth_processor, _depth_model

def unload_depth_model() -> None:
    """Frees the standalone depth-estimation model. It has no other owner and, unlike
    the ControlNet/SD pipelines, was never freed anywhere — once loaded it stayed
    resident in system RAM (it runs on CPU, not the GPU pipeline) for the rest of the
    process, compounding with whatever checkpoint loads next until the runtime ran out
    of RAM. Called from ControlNetManager.unload() so it shares the pipeline's own
    lifecycle instead of outliving it."""
    global _depth_processor, _depth_model
    if _depth_model is not None:
        _depth_processor = None
        _depth_model = None
        gc.collect()

def depth_preprocess(image: Image.Image) -> Image.Image:
    processor, model = _load_depth_model()
    inputs = processor(images=image, return_tensors="pt")
    with torch.no_grad():
        predicted_depth = model(**inputs).predicted_depth

    depth = torch.nn.functional.interpolate(
        predicted_depth.unsqueeze(1),
        size=image.size[::-1],
        mode="bicubic",
        align_corners=False,
    )[0, 0]
    depth = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)
    depth_array = (depth.numpy() * 255).astype(np.uint8)
    return Image.fromarray(depth_array).convert("RGB")

class ControlNetManager:
    """Separate from PipelineManager since it needs its own pipeline class
    (StableDiffusionControlNetPipeline) and a different reload trigger: the
    active set of ControlNet *types*, not just the checkpoint."""

    def __init__(self) -> None:
        self.device = get_device()
        self.dtype = get_dtype()
        self.pipe: StableDiffusionControlNetPipeline | None = None
        self.checkpoint_id: str | None = None
        self.active_types: list[str] = []
        self.lora_id: str = "none"

    @property
    def is_ready(self) -> bool:
        return self.pipe is not None

    def load(self, checkpoint_id: str, controlnet_types: list[str], on_status=None) -> None:
        on_status = on_status or (lambda _msg: None)
        if self.pipe is not None and self.checkpoint_id == checkpoint_id and self.active_types == controlnet_types:
            return

        if self.pipe is not None:
            self.unload()

        checkpoint = get_checkpoint(checkpoint_id)
        on_status(f"Loading {checkpoint.label} + ControlNet ({', '.join(controlnet_types)})...")
        # low_cpu_mem_usage=False avoids accelerate's meta-device load path, which on
        # some torch/accelerate combos (notably Colab) leaves .to(device) unable to copy.
        controlnets = [
            ControlNetModel.from_pretrained(CONTROLNET_REPOS[t], torch_dtype=self.dtype, low_cpu_mem_usage=False)
            for t in controlnet_types
        ]
        controlnet_arg = controlnets[0] if len(controlnets) == 1 else controlnets

        pipe = StableDiffusionControlNetPipeline.from_pretrained(
            checkpoint.repo_id, controlnet=controlnet_arg, torch_dtype=self.dtype, low_cpu_mem_usage=False
        ).to(self.device)
        pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

        # is_ready reads `self.pipe is not None` and is polled from another thread (the
        # state panel) without a lock — every other attribute it depends on must already
        # be correct before self.pipe flips non-None, not after. Build the pipeline into
        # a local first, then assign checkpoint_id/active_types/lora_id, and only then
        # self.pipe last — same fix as pipeline_manager.load_checkpoint.
        self.checkpoint_id = checkpoint_id
        self.active_types = controlnet_types
        self.lora_id = "none"  # a fresh pipeline carries no adapter
        self.pipe = pipe
        on_status("ControlNet pipeline ready.")

    def set_lora(self, lora_id: str, on_status=None) -> None:
        """Load/unload a style LoRA on the ControlNet pipeline — mirror of
        PipelineManager.set_lora, so a style can be combined with a reference image."""
        on_status = on_status or (lambda _msg: None)
        if self.pipe is None:
            raise RuntimeError("Load a ControlNet pipeline before selecting a LoRA.")
        if lora_id == self.lora_id:
            return
        if self.lora_id != "none":
            self.pipe.unload_lora_weights()
        lora = get_lora(lora_id)
        if lora.path is not None:
            local = os.path.join(APP_LORAS_DIR, f"{lora.id}-lora.safetensors")
            source = local if os.path.exists(local) else lora.path
            on_status(f"Loading LoRA {lora.label}...")
            # low_cpu_mem_usage=False: see the identical note in pipeline_manager.set_lora.
            self.pipe.load_lora_weights(source, low_cpu_mem_usage=False)
        self.lora_id = lora_id
        on_status(f"LoRA set to {lora.label}.")

    def unload(self) -> None:
        if self.pipe is not None:
            # Plain assignment, not `del self.pipe` — a concurrent reader of `is_ready`
            # (the state panel polls from another thread) would briefly see the
            # attribute missing entirely instead of just None. See pipeline_manager.py.
            self.pipe = None
            self.checkpoint_id = None
            self.active_types = []
            self.lora_id = "none"
            gc.collect()
            empty_cache(self.device)
        # Unconditional, not nested in the branch above: Process can load the depth
        # model without ever running a generation (self.pipe staying None), and this
        # is the only place that ever frees it.
        unload_depth_model()

    def generate(
        self,
        prompt: str,
        negative_prompt: str,
        control_images: list[Image.Image],
        conditioning_scales: list[float],
        cfg_scale: float,
        steps: int,
        seed: int,
        lora_scale: float | None = None,
    ) -> Image.Image:
        if self.pipe is None:
            raise RuntimeError("Load a ControlNet pipeline first.")

        control_image_arg = control_images[0] if len(control_images) == 1 else control_images
        # diffusers checks `isinstance(controlnet_conditioning_scale, float)` for the
        # single-ControlNet case and rejects ints / numpy scalars — cast explicitly.
        scales = [float(s) for s in conditioning_scales]
        scale_arg = scales[0] if len(scales) == 1 else scales
        gen_device = self.device if self.device != "mps" else "cpu"
        generator = torch.Generator(device=gen_device).manual_seed(seed)
        # Same knob as the plain pipeline: the LoRA weight rides on the UNet cross-
        # attention, so ControlNet residuals and the style adapter stack cleanly.
        cross_attention_kwargs = {"scale": lora_scale} if lora_scale is not None else None

        result = self.pipe(
            prompt=prompt,
            image=control_image_arg,
            negative_prompt=negative_prompt or None,
            controlnet_conditioning_scale=scale_arg,
            guidance_scale=cfg_scale,
            num_inference_steps=steps,
            generator=generator,
            cross_attention_kwargs=cross_attention_kwargs,
        )
        # See generation._generate_plain's identical check: the safety checker swaps a
        # flagged image for a plain black square and only ever says so via a console
        # warning otherwise.
        if getattr(result, "nsfw_content_detected", None) and result.nsfw_content_detected[0]:
            raise RuntimeError(
                "The safety checker flagged this result as potential NSFW content and blocked it "
                "(a black square, not a real image) — try a different prompt or seed."
            )
        return result.images[0]


### Single-image generation
_(`src/latent_studio/generation.py`)_

`GenerationParams` is the one struct describing everything needed to reproduce a single image (prompt, negative prompt, size, CFG, steps, seed, LoRA weight, and optionally ControlNet type(s)/scale(s)). `generate()` is the single dispatch point that turns a `GenerationParams` into an actual image: it routes to the plain SD1.5 pipeline, or to the ControlNet pipeline if `controlnet_types` is set — the UI and the sweep engine below both call this one function, so neither has to know which pipeline is actually running.

In [ ]:
#@title generation.py
"""Single-generation entry point. generate() dispatches to the plain SD1.5 path
or the ControlNet path and returns the image together with the metadata dict
that gets shown in the UI and embedded into the PNG."""

@dataclass
class GenerationParams:
    prompt: str
    negative_prompt: str
    width: int
    height: int
    cfg_scale: float
    steps: int
    seed: int
    lora_weight: float = 1.0
    controlnet_types: list[str] = field(default_factory=list)
    controlnet_scales: dict[str, float] = field(default_factory=dict)

def _effective_prompt(lora_id: str, prompt: str) -> str:
    """Auto-prepend the active LoRA's trigger word so the style actually fires
    without the user having to know/type it. No-op if there's no LoRA/trigger or
    the trigger is already present."""
    lora = get_lora(lora_id)
    if lora.path is None or not lora.trigger_word:
        return prompt
    if lora.trigger_word.lower() in prompt.lower():
        return prompt
    return f"{lora.trigger_word}, {prompt}".strip().rstrip(",").strip()

def _tag_hardware(metadata: dict, mgr) -> None:
    """Record the device + dtype the image was actually rendered on. Same seed and
    settings only reproduce exactly on the same backend — cuda/fp16 and mps/fp32
    diverge — so without this the metadata block can't uniquely identify its image."""
    metadata["device"] = mgr.device
    metadata["dtype"] = str(mgr.dtype).replace("torch.", "")

def _generate_plain(manager: PipelineManager, params: GenerationParams) -> tuple[Image.Image, str]:
    if not manager.is_ready:
        raise RuntimeError("No model loaded — select a checkpoint first.")

    # Only pass a scale if a LoRA is actually loaded (lora.path is not None).
    lora = get_lora(manager.lora_id)
    cross_attention_kwargs = {"scale": params.lora_weight} if lora.path is not None else None

    prompt = _effective_prompt(manager.lora_id, params.prompt)
    generator = manager.generator(params.seed)
    result = manager.pipe(
        prompt=prompt,
        negative_prompt=params.negative_prompt or None,
        width=params.width,
        height=params.height,
        guidance_scale=params.cfg_scale,
        num_inference_steps=params.steps,
        generator=generator,
        cross_attention_kwargs=cross_attention_kwargs,
    )
    # The safety checker replaces a flagged image with a plain black square and only
    # ever mentions it via a console warning — silently "succeeding" with that black
    # square would be far more confusing than a clear error in the state panel.
    if getattr(result, "nsfw_content_detected", None) and result.nsfw_content_detected[0]:
        raise RuntimeError(
            "The safety checker flagged this result as potential NSFW content and blocked it "
            "(a black square, not a real image) — try a different prompt or seed."
        )
    return result.images[0], prompt

def generate(
    manager: PipelineManager,
    controlnet_manager: ControlNetManager,
    params: GenerationParams,
    control_images: dict[str, Image.Image] | None = None,
) -> tuple[Image.Image, dict]:
    metadata = asdict(params)

    if params.controlnet_types:
        control_images = control_images or {}
        ordered_images = [control_images[t] for t in params.controlnet_types]
        ordered_scales = [params.controlnet_scales[t] for t in params.controlnet_types]
        # LoRA (patches the UNet) and ControlNet (injects residuals) are orthogonal and combine freely.
        lora = get_lora(controlnet_manager.lora_id)
        lora_scale = params.lora_weight if lora.path is not None else None
        effective_prompt = _effective_prompt(controlnet_manager.lora_id, params.prompt)
        image = controlnet_manager.generate(
            prompt=effective_prompt,
            negative_prompt=params.negative_prompt,
            control_images=ordered_images,
            conditioning_scales=ordered_scales,
            cfg_scale=params.cfg_scale,
            steps=params.steps,
            seed=params.seed,
            lora_scale=lora_scale,
        )
        checkpoint = get_checkpoint(controlnet_manager.checkpoint_id)
        metadata["checkpoint_id"] = checkpoint.id
        metadata["checkpoint_label"] = checkpoint.label
        metadata["lora_id"] = lora.id
        metadata["lora_label"] = lora.label
        if lora.path is None:
            metadata.pop("lora_weight", None)
        elif effective_prompt != params.prompt:
            metadata["effective_prompt"] = effective_prompt  # trigger word auto-added
        _tag_hardware(metadata, controlnet_manager)
    else:
        image, effective_prompt = _generate_plain(manager, params)
        checkpoint = get_checkpoint(manager.checkpoint_id)
        lora = get_lora(manager.lora_id)
        metadata["checkpoint_id"] = checkpoint.id
        metadata["checkpoint_label"] = checkpoint.label
        metadata["lora_id"] = lora.id
        metadata["lora_label"] = lora.label
        if lora.path is None:
            metadata.pop("lora_weight", None)
        elif effective_prompt != params.prompt:
            metadata["effective_prompt"] = effective_prompt  # trigger word auto-added
        metadata.pop("controlnet_types", None)
        metadata.pop("controlnet_scales", None)
        _tag_hardware(metadata, manager)

    return image, metadata


### Metadata embed/read
_(`src/latent_studio/metadata.py`)_

Every generated image's full settings are embedded directly into the PNG's own text chunk (`save_with_metadata`) and can be read back out of just the image file (`read_metadata`) — this is what makes 'reproduce this exact image from nothing but the file itself' possible, which is one of the mandatory live-demo steps.

In [ ]:
#@title metadata.py
"""Embed/read generation metadata on PNGs. This is what makes the mandatory
"reproduce an example image from its own metadata" demo step possible: the
settings travel with the file, not just in a separate log."""

METADATA_KEY = "latent_studio"

def save_with_metadata(image: Image.Image, metadata: dict, path: str) -> None:
    png_info = PngInfo()
    png_info.add_text(METADATA_KEY, json.dumps(metadata))
    image.save(path, pnginfo=png_info)

def read_metadata(path: str) -> dict | None:
    with Image.open(path) as image:
        image.load()  # tEXt chunks after IDAT are only parsed once the image is fully read
        raw = image.info.get(METADATA_KEY)
    if raw is None:
        return None
    return json.loads(raw)

def format_metadata(metadata: dict) -> str:
    """Human-readable block for display under a generated image in the Gradio UI."""
    lines = [f"**{key}:** {value}" for key, value in metadata.items()]
    return "\n".join(lines)


### Grid compositing + parameter sweeps
_(`src/latent_studio/grids.py`)_

`make_grid_image` composites a list of generated images into one labeled grid PNG. `sweep()` is a generic 1-2 dimension parameter sweep: give it `dim1=(field_name, (min, max), count)` — any of `cfg_scale`, `steps`, `lora_weight`, or `seed` — and optionally a `dim2` for a second axis, and it generates every combination and lays them out as a grid (`dim1` across columns, `dim2` across rows). For `seed` the pair means `(start, step)` rather than `(from, to)`: interpolating between two seeds is meaningless, but 'every 10th seed' isn't. This engine backs the Compare mode in the app below — which is also what generated the submitted Parameter Atlas.

In [ ]:
#@title grids.py
"""Grid compositing + a generic parameter-sweep engine. This is the engine behind
the app's Compare mode, which is also how the Parameter Atlas is produced —
generate_grid()/sweep() take a plain callable rather than a PipelineManager
directly, so the same engine drives both the plain and the ControlNet path."""

GenerateFn = Callable[[GenerationParams], tuple[Image.Image, dict]]

# Which GenerationParams fields can be swept, and how their values are typed.
# "seed" is handled separately (start + step * i, not an interpolated range) since
# "the range between two seeds" isn't meaningful.
SWEEPABLE_FIELDS: dict[str, type] = {
    "cfg_scale": float,
    "steps": int,
    "lora_weight": float,
}

# Cell labels are burned into the grid PNG, which ends up in the docs and in the
# live demo — so they carry the names the UI uses, not the field names.
FIELD_LABELS: dict[str, str] = {
    "cfg_scale": "prompt strength",
    "steps": "detail",
    "lora_weight": "style strength",
    "seed": "seed",
}

def interpolate_range(value_range: tuple[float, float], count: int, value_type=float) -> list:
    start, end = value_range
    if count <= 1:
        values = [start]
    else:
        step = (end - start) / (count - 1)
        values = [start + step * i for i in range(count)]

    if value_type is int:
        return [max(1, int(round(v))) for v in values]
    return [round(float(v), 2) for v in values]

def make_grid_image(
    images: list[Image.Image | None],
    width: int,
    height: int,
    grid_rows: int,
    grid_cols: int,
    labels: list[str] | None = None,
) -> Image.Image:
    grid_image = Image.new("RGB", (width * grid_cols, height * grid_rows), color=(32, 32, 32))
    draw = ImageDraw.Draw(grid_image)
    font = ImageFont.load_default()

    for idx in range(grid_rows * grid_cols):
        row, col = divmod(idx, grid_cols)
        x, y = col * width, row * height

        image = images[idx] if idx < len(images) else None
        if image is not None:
            grid_image.paste(image.resize((width, height)), (x, y))
        else:
            draw.rectangle([x, y, x + width - 1, y + height - 1], outline=(90, 90, 90), width=2)
            draw.text((x + 12, y + 12), "missing", fill=(220, 220, 220), font=font)

        if labels and idx < len(labels):
            label = labels[idx]
            bbox = draw.textbbox((0, 0), label, font=font)
            label_w, label_h = bbox[2] - bbox[0], bbox[3] - bbox[1]
            padding = 6
            draw.rectangle(
                [x, y + height - label_h - padding * 2, x + label_w + padding * 2, y + height],
                fill=(0, 0, 0),
            )
            draw.text((x + padding, y + height - label_h - padding), label, fill=(255, 255, 255), font=font)

    return grid_image

def generate_grid(
    generate_fn: GenerateFn,
    params_list: list[GenerationParams],
    grid_rows: int,
    grid_cols: int,
    labels: list[str] | None = None,
    progress: Callable[[int, int], None] | None = None,
) -> tuple[Image.Image, list[dict]]:
    """Runs each GenerationParams through generate_fn and composites the
    results. len(params_list) must equal grid_rows * grid_cols. If given,
    `progress(index, total)` is called before each cell — that's what drives the
    app's "Image 3 of 20" bar (progress.ProgressTracker.on_image)."""
    images: list[Image.Image] = []
    metadata_list: list[dict] = []
    total = len(params_list)
    for i, params in enumerate(params_list):
        if progress is not None:
            progress(i, total)
        image, metadata = generate_fn(params)
        images.append(image)
        metadata_list.append(metadata)

    width, height = params_list[0].width, params_list[0].height
    grid_image = make_grid_image(images, width, height, grid_rows, grid_cols, labels)
    return grid_image, metadata_list

def _sweep_values(field_name: str, value_range: tuple[float, float], count: int) -> list:
    if field_name == "seed":
        # For seeds the pair is (start, step), not (from, to) — interpolating
        # between two seeds is meaningless, but "every 10th seed" isn't.
        start, step = int(value_range[0]), int(value_range[1])
        return [start + i * step for i in range(count)]
    return interpolate_range(value_range, count, SWEEPABLE_FIELDS[field_name])

def sweep(
    generate_fn: GenerateFn,
    base_params: GenerationParams,
    dim1: tuple[str, tuple[float, float], int],
    dim2: tuple[str, tuple[float, float], int] | None = None,
    shape: tuple[int, int] | None = None,
    progress: Callable[[int, int], None] | None = None,
) -> tuple[Image.Image, list[dict]]:
    """Generic 1-2 dimension sweep over any field in SWEEPABLE_FIELDS plus "seed".
    Each dim is (field_name, value_range, count) — (from, to) for numeric fields,
    (start, step) for "seed". One dimension -> a row of `count` cells (override
    with `shape=(rows, cols)`); two dimensions -> dim1 across columns, dim2 down
    rows."""
    name1, range1, count1 = dim1
    values1 = _sweep_values(name1, range1, count1)

    if dim2 is None:
        params_list = [replace(base_params, **{name1: v}) for v in values1]
        labels = [f"{FIELD_LABELS[name1]} {v}" for v in values1]
        rows, cols = shape or (1, count1)
        return generate_grid(generate_fn, params_list, rows, cols, labels, progress)

    name2, range2, count2 = dim2
    values2 = _sweep_values(name2, range2, count2)
    params_list = [
        replace(base_params, **{name1: v1, name2: v2}) for v2 in values2 for v1 in values1
    ]
    labels = [
        f"{FIELD_LABELS[name1]} {v1} | {FIELD_LABELS[name2]} {v2}"
        for v2 in values2
        for v1 in values1
    ]
    return generate_grid(generate_fn, params_list, count2, count1, labels, progress)


### Icons
_(`src/latent_studio/icons.py`)_

One monochrome stroke-icon set for the whole UI, so nothing in it is an emoji. Buttons get theirs as a `data:` URI (Gradio renders a button icon as an `<img>`, and this notebook has no files to point one at); the live state panel gets inline `<svg>`, where `currentColor` still works.

In [ ]:
#@title icons.py
"""One monochrome stroke-icon set for the whole UI, replacing emoji. Button icons
are baked as a `data:` URI (no file written to disk); the state panel uses inline
`<svg>` instead, where `currentColor` works."""

# Icon color for the two button kinds. The secondary tone is a mid warm grey that
# stays legible on both the light and the dark button fill.
ICON_ON_PRIMARY = "#ffffff"
ICON_ON_SECONDARY = "#8a817c"

# 24x24 viewBox, stroke-drawn. {c} is the stroke/fill color, substituted per use.
_MARKUP: dict[str, str] = {
    "sparkles": (
        '<path d="M12 3.5l1.9 5.1 5.1 1.9-5.1 1.9-1.9 5.1-1.9-5.1L5 10.5l5.1-1.9z"/>'
        '<path d="M18.5 15.5l.7 1.8 1.8.7-1.8.7-.7 1.8-.7-1.8-1.8-.7 1.8-.7z"/>'
    ),
    "download": '<path d="M12 3v12"/><path d="M7 11l5 5 5-5"/><path d="M4 20h16"/>',
    "upload": '<path d="M12 15V3"/><path d="M7 8l5-5 5 5"/><path d="M4 20h16"/>',
    "reuse": '<path d="M3 3v6h6"/><path d="M21 12A9 9 0 0 0 6 5.3L3 9"/>',
    "copy": (
        '<rect x="9" y="9" width="12" height="12" rx="2"/>'
        '<path d="M5 15a2 2 0 0 1-2-2V5a2 2 0 0 1 2-2h8a2 2 0 0 1 2 2"/>'
    ),
    "dice": (
        '<rect x="3" y="3" width="18" height="18" rx="3"/>'
        '<circle cx="8.5" cy="8.5" r="1.3" fill="{c}" stroke="none"/>'
        '<circle cx="12" cy="12" r="1.3" fill="{c}" stroke="none"/>'
        '<circle cx="15.5" cy="15.5" r="1.3" fill="{c}" stroke="none"/>'
    ),
    "check":'<circle cx="12" cy="12" r="9"/><path d="M8.5 12.5l2.5 2.5 4.5-5"/>',
    "alert": (
        '<path d="M10.3 4.3 2.6 17.5A2 2 0 0 0 4.3 20.5h15.4a2 2 0 0 0 1.7-3L13.7 4.3a2 2 0 0 0-3.4 0z"/>'
        '<path d="M12 9.5v4"/><path d="M12 17h.01"/>'
    ),
}

def svg(name: str, size: int = 18, color: str = "currentColor") -> str:
    """Inline <svg> markup — for gr.HTML, where currentColor works."""
    return (
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{size}" height="{size}" '
        f'viewBox="0 0 24 24" fill="none" stroke="{color}" stroke-width="1.75" '
        f'stroke-linecap="round" stroke-linejoin="round" '
        f'style="flex:none;vertical-align:middle">{_MARKUP[name].format(c=color)}</svg>'
    )

def button_icon(name: str, color: str = ICON_ON_SECONDARY) -> dict:
    """Icon value for gr.Button/gr.UploadButton/gr.DownloadButton(icon=...) — a data:
    URI in FileData clothing. The dict form matters: Block.serve_static_file() passes a
    dict straight through untouched, but treats a bare string as a literal filesystem
    path and tries to cache it as a file — which silently breaks for a data: URI. (Every
    button's icon= type hint says `str`, but the dict is what actually renders.)"""
    uri = "data:image/svg+xml;utf8," + quote(svg(name, size=20, color=color))
    return {"path": uri, "url": uri}


### Live progress
_(`src/latent_studio/progress.py`)_

What puts a real progress bar — with an ETA — in front of the user instead of a spinner. A diffusers call blocks, so the app runs it on a worker thread and polls this `ProgressTracker`. The numbers come for free: diffusers' denoising loop and huggingface_hub's downloader are both **tqdm** subclasses, so patching `tqdm` once (`track_tqdm`) reports megabytes while a model downloads and steps while an image renders, with no callback plumbed through the pipeline. A second bar counts images ('image 7 of 25') when a comparison grid is running.

In [ ]:
#@title progress.py
"""Live progress for the state panel: two bars and an ETA, fed by patching tqdm
(diffusers' denoise loop and huggingface_hub's downloader are both tqdm
subclasses) rather than a callback threaded through the pipeline."""

_ACCENT = "var(--color-accent, #b4552f)"
_TRACK = "var(--background-fill-secondary, rgba(120,113,108,.18))"

def _duration(seconds: float | None) -> str:
    if seconds is None or seconds != seconds or seconds in (float("inf"), 0) or seconds < 0:
        return "—"
    total = int(seconds)
    if total >= 3600:
        return f"{total // 3600}h {(total % 3600) // 60:02d}m"
    if total >= 60:
        return f"{total // 60}m {total % 60:02d}s"
    return f"{total}s"

def _amount(value: float, unit: str) -> str:
    if unit != "B":
        return f"{int(value)}"
    for suffix in ("B", "KB", "MB", "GB"):
        if value < 1024 or suffix == "GB":
            return f"{value:.0f} {suffix}" if suffix == "B" else f"{value:.1f} {suffix}"
        value /= 1024
    return f"{value:.1f} GB"

def _bar(label: str, right: str, fraction: float | None) -> str:
    # fraction None = we know something is running but not how far along.
    width = 100.0 if fraction is None else max(0.0, min(1.0, fraction)) * 100
    opacity = ".35" if fraction is None else "1"
    return (
        '<div style="margin-top:.6rem">'
        '<div style="display:flex;justify-content:space-between;gap:1rem;'
        'font-family:var(--font-mono);font-size:.8rem;opacity:.75;margin-bottom:.3rem">'
        f"<span>{label}</span><span>{right}</span></div>"
        f'<div style="height:7px;border-radius:99px;background:{_TRACK};overflow:hidden">'
        f'<div style="height:100%;width:{width:.1f}%;opacity:{opacity};background:{_ACCENT};'
        'border-radius:99px;transition:width .25s linear"></div></div></div>'
    )

def status_html(icon: str, text: str) -> str:
    """The panel's resting states: ready / done / error."""
    return (
        '<div style="display:flex;align-items:center;gap:.5rem;line-height:1.4">'
        f"{svg(icon)}<span>{text}</span></div>"
    )

def _phase_line(text: str) -> str:
    """The live phase, plain text, no icon — the spinner was the redundant bit (the
    Generate button already shows the coarse state); dropping the text too left the
    panel fully blank during a plain generate, which read as broken rather than idle."""
    return f'<div style="line-height:1.4">{text}</div>'

class ProgressTracker:
    """Shared state between the worker thread (writer) and the UI loop (reader)."""

    def __init__(self) -> None:
        self._lock = threading.Lock()
        self.phase = "Starting…"
        self.button = "Working…"  # the coarse state, shown on the Generate button
        self._bars: dict[int, dict] = {}  # live tqdm bars, insertion-ordered
        self.image_index = 0
        self.image_total = 1
        self._images_started = time.monotonic()

    # -- written by the worker thread ------------------------------------
    def set_phase(self, phase: str, button: str, image_total: int = 1) -> None:
        with self._lock:
            self.phase = phase
            self.button = button
            self.image_index = 0
            self.image_total = image_total
            self._images_started = time.monotonic()

    def on_image(self, index: int, total: int) -> None:
        """grids.generate_grid() calls this before each cell."""
        with self._lock:
            self.image_index = index
            self.image_total = total

    def bar_open(self, key: int, total, desc: str, unit: str) -> None:
        with self._lock:
            self._bars[key] = {
                "n": 0, "total": total, "desc": desc, "unit": unit, "t0": time.monotonic()
            }

    def bar_update(self, key: int, n: float, total) -> None:
        with self._lock:
            bar = self._bars.get(key)
            if bar is not None:
                bar["n"], bar["total"] = n, total

    def bar_close(self, key: int) -> None:
        with self._lock:
            self._bars.pop(key, None)

    # -- read by the UI loop ---------------------------------------------
    def html(self) -> str:
        """The live phase as plain text (no spinner — the Generate button already
        shows the coarse state, so the icon was the redundant bit, not the words),
        plus whatever progress bars are actually running."""
        with self._lock:
            phase, index, total = self.phase, self.image_index, self.image_total
            started = self._images_started
            # The outermost still-open bar is the meaningful one (e.g. "Fetching N
            # files" rather than one of its per-file children).
            bar = next(iter(self._bars.values()), None)
            bar = dict(bar) if bar else None

        blocks = [_phase_line(phase)]
        inner_fraction = 0.0

        if bar:
            n, bar_total, unit = bar["n"], bar["total"], bar["unit"]
            elapsed = time.monotonic() - bar["t0"]
            if bar_total:
                inner_fraction = n / bar_total
                eta = (bar_total - n) / (n / elapsed) if n and elapsed else None
                right = f"{_amount(n, unit)} / {_amount(bar_total, unit)} · {_duration(eta)} left"
                blocks.append(_bar(bar["desc"] or "Working", right, inner_fraction))
            else:
                blocks.append(_bar(bar["desc"] or "Working", _duration(elapsed), None))

        if total > 1:
            done = index + inner_fraction  # count the image in flight, so the bar creeps
            elapsed = time.monotonic() - started
            eta = (total - done) / (done / elapsed) if done and elapsed else None
            blocks.append(
                _bar(f"Image {min(index + 1, total)} of {total}", f"{_duration(eta)} left", done / total)
            )

        return "".join(blocks)

@contextmanager
def track_tqdm(tracker: ProgressTracker):
    """Routes every tqdm bar created while this is active into `tracker`, by
    patching the shared tqdm base class (global, but the app is single-user)."""
    original = (_std_tqdm.__init__, _std_tqdm.update, _std_tqdm.close)

    def patched_init(self, *args, **kwargs):
        original[0](self, *args, **kwargs)
        if not getattr(self, "disable", False):
            tracker.bar_open(
                id(self), self.total, getattr(self, "desc", "") or "", getattr(self, "unit", "it")
            )

    def patched_update(self, n=1):
        result = original[1](self, n)
        if not getattr(self, "disable", False):
            tracker.bar_update(id(self), self.n, self.total)
        return result

    def patched_close(self):
        tracker.bar_close(id(self))
        return original[2](self)

    _std_tqdm.__init__, _std_tqdm.update, _std_tqdm.close = patched_init, patched_update, patched_close
    try:
        yield tracker
    finally:
        _std_tqdm.__init__, _std_tqdm.update, _std_tqdm.close = original


### Design tokens: theme, copy, sweep ranges
_(`src/latent_studio/design_tokens.py`)_

Pure data — the Gradio `THEME`, every string of UI copy, and the Compare tab's per-setting ranges. Nothing here calls `gr.Blocks`, so a wording or range tweak never touches app logic. Two kinds of copy are kept apart on purpose: a `DESC_*` is the static explanation rendered as Markdown under a control's title; a `STATUS_*` is the live reason a control is greyed out, and is the *only* thing a control's own `info=` ever carries (empty when the control is active) — `callbacks.gating_updates()` swaps a STATUS in and out. `SweepSpec` gives each Compare-tab setting its own slider range, default sweep range and image count — one global default would be meaningless across four differently-scaled fields (a CFG range of 3-15 vs. a seed range in the billions). `THEME` drives colour entirely through Gradio's own hue system (`primary_hue` — swap it to re-skin); the `.set()` override zeros the border/background on plain component-level blocks, while `gr.Column(variant="panel")` still gets its box from its own separate `panel_*` tokens. `gr.Group()` is deliberately not used anywhere in app.py: its compiled CSS reads its background from `--border-color-primary`, not `--block-background-fill`, and that variable is also the real border color for dozens of unrelated elements (gallery thumbnails, focus rings) — no clean theme lever exists to switch it off without side effects elsewhere, so plain title+description+control spacing (via `_heading()`) replaces it instead. `THEME` is passed to `launch()` in the app.py section below, not the `Blocks` constructor (Gradio 6 moved it there).

In [ ]:
#@title design_tokens.py
"""Design tokens: theme, the copy shown in the UI, and the Compare tab's sweep
ranges. Pure data — no gr.Blocks calls — so tweaking a range or a wording never
touches app logic."""

WIDTH = HEIGHT = 512
MAX_SEED = 2**32 - 1
MAX_IMAGES = 100  # a comparison beyond this is a runaway, not a study

# The Prompt box starts pre-filled with a random example. Content only, no medium/
# style words (no "35mm film", "painterly", "cyberpunk") — a style LoRA supplies the
# look, so the wording is chosen to give it clean subjects to work with.
EXAMPLE_PROMPTS = [
    "a great wave breaking over small fishing boats, seen from the shore",
    "a lone mountain rising above layered clouds at sunrise",
    "a wooden bridge crossing a river in a rain shower",
    "travellers resting under a large pine tree beside a dirt road",
    "a small boat drifting on a calm lake surrounded by lily pads",
    "a field of poppies stretching toward a distant windmill",
    "a rabbit sitting still in short grass, seen up close",
    "an old man's face lit from one side, deep shadow behind him",
    "a ship battling a storm at sea, waves crashing over the deck",
    "a garden path lined with irises leading to a small footbridge",
    "a hand holding a bundle of wildflowers against a plain background",
    "a harbor at dusk with fishing boats returning to shore",
    "a quiet village street after snowfall, footprints in the snow",
    "a hawk perched on a bare branch against a pale sky",
    "an old stone watermill beside a rushing stream",
    "a woman reading a letter by a window, soft daylight",
]

CONTROLNET_CHOICES = [("Off", "off"), ("Follow the outlines", "canny"), ("Follow the depth", "depth")]

# DESC_* = the static Markdown shown under a control's title. STATUS_* = the live
# reason a control is greyed out, swapped into its info= by callbacks.gating_updates.
DESC_PROMPT = "What should be in the picture. Naming a subject, a light and a look works best."
DESC_AVOID = (
    "Anything to keep out of the picture — leave it empty if nothing bothers you. "
    "(This is the negative prompt.)"
)
DESC_MODEL = (
    "The engine that paints. **Stable Diffusion 1.5** is the plain original, a neutral "
    "all-rounder. **DreamShaper 8** is a community retrain blending stylized and photoreal "
    "looks. **epiCRealism** is a community retrain pushed hard toward photorealism. "
    "Switching reloads the model, which takes a moment."
)
DESC_STYLE = (
    "An artist's hand laid over the model. Each is a **LoRA** trained for this project on that "
    "artist's public-domain works from the Art Institute of Chicago, then scored against the "
    "real paintings before it was allowed in here."
)
DESC_STYLE_STRENGTH = (
    "How far the style is pushed. 0 is the plain model, 1 is the style as it was trained, and "
    "above ~1.2 it usually burns — colours go flat and shapes fall apart."
)
DESC_CFG = (
    "Called guidance scale (CFG). How literally the model takes your prompt. 1–5 wanders off "
    "and invents; 7–9 is the sweet spot; above ~14 it forces the prompt through and the image "
    "turns harsh and over-contrasted."
)
DESC_STEPS = (
    "Called inference steps. How many passes the model makes to sharpen the image. Under 15 it "
    "stays smudgy, 25–35 is plenty, past ~50 you mostly pay time for nothing. Doubling the "
    "steps doubles the wait."
)
DESC_SEED = (
    "The random starting point behind the image. A new one every click gives you variety; "
    "fix it and the same settings always return the exact same image — that is how a "
    "result is reproduced."
)
DESC_REFERENCE = (
    "Optional, and called ControlNet. Give it a picture and the result keeps that picture's "
    "composition — either its outlines, or how near and far its parts are. Your prompt still "
    "decides what things are made of. **Experimental** — still being tested, so treat it as a "
    "feature to use at your own risk rather than a finished control."
)
DESC_REFERENCE_SCALE = (
    "How tightly the result sticks to the reference. Around 0.4 it is a loose suggestion, 1.0 "
    "follows the shapes closely, above ~1.5 it traces them and ignores your prompt."
)
DESC_COMPARE = (
    "Generate a grid instead of one image — the fastest way to see what a setting "
    "actually does, laid out side by side."
)
DESC_ADVANCED = (
    "Adds a panel with the full settings of the current image and a button to reuse them, "
    "and unlocks the experimental Reference image tab."
)
DESC_PRELOAD = (
    "Each model downloads the first time you use it. Pull **all** of them now — handy right "
    "before a live demo, so the first switch isn't a wait. It only fills the cache; nothing is "
    "kept in memory."
)

STATUS_SWEPT = "Being compared right now — set its range in the Compare tab."
STATUS_STYLE_OFF = "Pick a style first."
STATUS_SEED_RANDOM = "Switch to a fixed seed to type your own."
STATUS_REFERENCE_OFF = "Pick a reference type first."

@dataclass(frozen=True)
class SweepSpec:
    """One Compare-tab setting: its slider range, default sweep range, and image count."""

    label: str
    minimum: float
    maximum: float
    step: float
    start: float
    end: float
    count: int
    max_count: int
    info: str

# Chosen per setting — one global default range/count would be meaningless across
# four very differently-scaled fields (a CFG range of 3-15 vs. a seed range in the
# billions).
SWEEP_SPECS: dict[str, SweepSpec] = {
    "cfg_scale": SweepSpec(
        "Prompt strength", 1.0, 20.0, 0.05, 3.0, 15.0, 5, 10,
        "Compares how literally the prompt is taken. Low values drift, high values "
        "get harsh — the useful range to look at is roughly 3 to 15.",
    ),
    "steps": SweepSpec(
        "Detail", 1, 100, 1, 10, 50, 5, 10,
        "Compares how much refinement the image gets. The interesting part is the "
        "low end: the difference between 10 and 30 is large, between 50 and 100 tiny.",
    ),
    "lora_weight": SweepSpec(
        "Style strength", 0.0, 1.5, 0.05, 0.0, 1.0, 5, 10,
        "Compares how hard the style is pushed. Sweep 0 → 1 to see it take hold, or "
        "past 1.2 to find the point where it burns.",
    ),
    "seed": SweepSpec(
        "Seed", 0, MAX_SEED, 1, 1312, 1, 9, 25,
        "Compares different random starting points with everything else held still — "
        "this is what shows you the spread a prompt can produce.",
    ),
}
SWEEP_CHOICES = [(spec.label, key) for key, spec in SWEEP_SPECS.items()]

# Colour comes entirely from Gradio's hue system (primary_hue); the .set() below is
# structural only. Passed to launch(), not the Blocks constructor (Gradio 6 moved it).
THEME = gr.themes.Base(
    primary_hue="rose",
    secondary_hue="stone",
    neutral_hue="stone",
    text_size="lg",
    spacing_size="lg",
    radius_size="xxl",
    font=[gr.themes.GoogleFont("Space Grotesk"), "ui-sans-serif", "system-ui", "sans-serif"],
    font_mono=[gr.themes.GoogleFont("Space Mono"), "ui-monospace", "Consolas", "monospace"],
).set(
    # Zeros the border/background on plain component-level blocks (Radio/Slider/etc.
    # already go container=False in app.py, so this mostly matters for anything that
    # doesn't). Deliberately not used to fight gr.Group()'s own box — its background
    # comes from a separate, widely-shared Gradio CSS variable with no clean lever —
    # app.py doesn't use gr.Group() at all for that reason; see _heading()'s docstring.
    block_background_fill="transparent",
    block_background_fill_dark="transparent",
    block_border_width="0px",
    block_border_width_dark="0px",
)


### Metadata display + downloads
_(`src/latent_studio/metadata_view.py`)_

Turns a generation result into what the user sees and can keep. A single image's metadata is a plain dict; a comparison grid's is `{"cells": [...], "compare": {...}}` — one metadata dict per cell plus the exact Compare-tab config (field/range/count per axis) that produced it. `_cells()`/`_compare_spec()` are the only things that read that shape directly; every other function goes through them, including for an older bare-list grid export with no stored sweep config, which `_compare_spec` reports as `None`. `settings_rows`/`settings_html`/`settings_json` read the cells — a swept field shows 'varies across the grid' rather than silently displaying the first cell's value — into the Advanced column and the downloadable JSON. `metadata_to_control_values` is the 'reuse these settings' path (also used by Import): it collapses ControlNet back to its single-select value, forces the seed mode to Fixed, and — new — restores Compare's field1/field2/Start/End/Steps from the stored `compare` block (or resets Compare to off if there isn't one, rather than leaving a stale sweep silently active). A pair of 'suppress the next reset' flags rides along in the same return tuple: Start/End/Steps have their own field-change cascade that re-ranges them to that field's generic default the instant field1/field2 changes, which would otherwise immediately clobber the exact values just restored here (see callbacks.dim_updates's `suppress` parameter). `_prepare_download_files` builds — and caches on the history entry — the PNG + settings-JSON pair a download button hands out. It's cached, and both download buttons carry their value reactively (set whenever generation finishes or a gallery thumbnail is selected) rather than building the file inside their own click handler, because a `gr.DownloadButton` that builds on click serves the *previous* click's file and lags one step behind.

In [ ]:
#@title metadata_view.py
"""Turns a generation result into what the user sees and downloads: the settings
list, the settings JSON, and the PNG + JSON files behind the download buttons."""

# (metadata key, human label) — the order they are shown in.
SETTINGS_ROWS = [
    ("checkpoint_label", "Model"),
    ("lora_label", "Style"),
    ("lora_weight", "Style strength"),
    ("prompt", "Prompt"),
    ("effective_prompt", "Prompt sent"),
    ("negative_prompt", "Avoid"),
    ("seed", "Seed"),
    ("cfg_scale", "Prompt strength"),
    ("steps", "Detail"),
]

def _cells(metadata) -> list[dict]:
    # A comparison grid stores {"cells": [...], "compare": {...}} (see _compare_spec);
    # a bare list is an older grid export with no stored sweep config. A single image
    # stores one plain dict.
    if isinstance(metadata, dict) and "cells" in metadata:
        return metadata["cells"]
    if isinstance(metadata, list):
        return metadata
    return [metadata] if metadata else []

def _compare_spec(metadata) -> dict | None:
    """The exact Compare-tab configuration that produced this grid (field/range/count
    per axis), if this generation came from Compare mode and stored one. None for a
    single image, or for an older bare-list grid export with no stored sweep config —
    either way the caller should reset Compare to off rather than guess at a sweep."""
    if isinstance(metadata, dict) and "compare" in metadata:
        return metadata["compare"]
    return None

def effective_prompt_of(metadata) -> str:
    cells = _cells(metadata)
    if not cells:
        return ""
    first = cells[0]
    return first.get("effective_prompt") or first.get("prompt") or ""

def _mono_box(text: str, center: bool = False) -> str:
    """Same monospace-panel look as settings_html, for a single line of text."""
    align = "text-align:center;" if center else ""
    return (
        f'<div style="font-family:var(--font-mono);font-size:.8rem;line-height:1.6;{align}'
        "background:var(--background-fill-secondary);border-radius:var(--radius-md);"
        f'padding:1rem;overflow-x:auto;word-break:break-word">{html.escape(text)}</div>'
    )

def prompt_used_html(metadata) -> str:
    """Centered like a caption under the gallery image, unlike settings_html's grid."""
    cells = _cells(metadata)
    if not cells:
        return _mono_box("", center=True)
    return _mono_box(effective_prompt_of(metadata), center=True)

def settings_rows(metadata) -> list[tuple[str, str]]:
    cells = _cells(metadata)
    if not cells:
        return []
    first = cells[0]
    rows: list[tuple[str, str]] = []
    for key, label in SETTINGS_ROWS:
        if key not in first:
            continue
        # In a grid, a swept setting differs cell to cell — say so instead of
        # silently showing the first cell's value.
        varies = len({json.dumps(cell.get(key), sort_keys=True) for cell in cells}) > 1
        value = "varies across the grid" if varies else first[key]
        rows.append((label, str(value) if str(value).strip() else "—"))
    rows.append(("Size", f"{first.get('width', WIDTH)} × {first.get('height', HEIGHT)}"))
    if first.get("controlnet_types"):
        rows.append(("Reference", ", ".join(first["controlnet_types"])))
        rows.append(("Reference strength", str(first.get("controlnet_scales", {}))))
    # Device + dtype decide reproducibility — the same seed only matches on the
    # same backend (cuda/fp16 vs mps/fp32 diverge).
    if first.get("device"):
        dtype = first.get("dtype", "")
        rows.append(("Rendered on", f"{first['device']}" + (f" · {dtype}" if dtype else "")))
    return rows

def settings_html(metadata) -> str:
    rows = settings_rows(metadata)
    if not rows:
        return '<p style="opacity:.6">Generate an image to see its settings.</p>'
    cells = "".join(
        f'<span style="opacity:.55;white-space:nowrap">{html.escape(label)}</span>'
        f'<span style="word-break:break-word">{html.escape(value)}</span>'
        for label, value in rows
    )
    return (
        '<div style="font-family:var(--font-mono);font-size:.8rem;line-height:1.6;'
        "background:var(--background-fill-secondary);border-radius:var(--radius-md);"
        'padding:1rem;overflow-x:auto">'
        '<div style="display:grid;grid-template-columns:auto 1fr;gap:.5rem .9rem">'
        f"{cells}</div></div>"
    )

def settings_json(metadata) -> str:
    return json.dumps(metadata or {}, indent=2, ensure_ascii=False)

def metadata_to_control_values(metadata) -> tuple:
    """Metadata dict -> the control values for "reuse these settings" (also used by
    Import). ControlNet collapses back to its single-select value; the seed mode
    switches to Fixed so reusing settings and then rolling Random can't silently change
    the seed.

    Compare's field1/field2/Start/End/Steps are restored too, from the "compare" block
    a grid export now carries (see callbacks.on_generate) — the exact sweep config used
    to produce it, not a reconstruction guessed from the cells' bare values (ambiguous:
    which field was the fast axis vs. the slow one isn't recoverable from values alone).
    A single image, or an older bare-list grid export with no stored sweep config, has
    no `compare` block — Compare resets to off in that case rather than leaving a stale
    sweep silently active (`_compare_spec` returns None for both).

    The trailing pair is a pair of "suppress the next reset" flags: Start/End/Steps
    have their own `field.change()` cascade that re-ranges them to that field's generic
    default the instant `field1`/`field2` changes — which would otherwise immediately
    clobber the exact values just restored here. `True` only when there's a real sweep
    to protect; `False` when resetting to off, where that cascade's normal (disabling)
    behavior is already exactly what's wanted."""
    cells = _cells(metadata)
    first = cells[0] if cells else {}
    types = first.get("controlnet_types", []) or []
    scales = first.get("controlnet_scales", {}) or {}
    cn_select = types[0] if types else "off"

    # Registry ids, not just missing keys: a file can name a checkpoint/style that
    # existed when it was exported but has since been retired from the roster (e.g. a
    # LoRA cut from the ship list) — gr.Radio validates a restored value against its
    # *current* choices and crashes with an uncaught popup otherwise, same failure mode
    # the sweep-bounds fix above addresses for Compare. Fall back to the default instead.
    checkpoint_id = first.get("checkpoint_id", DEFAULT_CHECKPOINT_ID)
    if checkpoint_id not in {cp.id for cp in CHECKPOINTS}:
        checkpoint_id = DEFAULT_CHECKPOINT_ID
    lora_id = first.get("lora_id", DEFAULT_LORA_ID)
    if lora_id not in {lora.id for lora in LORAS}:
        lora_id = DEFAULT_LORA_ID

    compare = _compare_spec(metadata)
    if compare:
        field1, field2 = compare.get("field1", "off"), compare.get("field2", "off")
        if field1 != "off" and field1 not in SWEEP_SPECS:
            field1 = "off"
        if field2 != "off" and field2 not in SWEEP_SPECS:
            field2 = "off"

        def _u(value):
            return gr.update(value=value) if value is not None else gr.update()

        def _count(value, field):
            # Clamped to that field's own max_count, not just passed through: this can
            # come from a hand-edited JSON or an export from an older app version, and
            # dim_updates() is about to realign the Number's maximum to this same field's
            # spec right after — an out-of-range value here would crash Gradio's own
            # bounds check instead of erroring gracefully.
            field_spec = SWEEP_SPECS.get(field)
            return None if value is None or field_spec is None else max(2, min(int(value), field_spec.max_count))

        from1_u, to1_u = _u(compare.get("from1")), _u(compare.get("to1"))
        from2_u, to2_u = _u(compare.get("from2")), _u(compare.get("to2"))
        count1_u = _u(_count(compare.get("count1"), field1))
        count2_u = _u(_count(compare.get("count2"), field2))
        suppress1 = suppress2 = True
    else:
        field1 = field2 = "off"
        from1_u = to1_u = count1_u = from2_u = to2_u = count2_u = gr.update()
        suppress1 = suppress2 = False

    return (
        checkpoint_id,
        lora_id,
        first.get("lora_weight", 1.0),
        first.get("prompt", ""),
        first.get("negative_prompt", ""),
        first.get("cfg_scale", 7.5),
        first.get("steps", 30),
        "fixed",
        first.get("seed", 0),
        cn_select,
        scales.get(cn_select, 1.0) if cn_select != "off" else 1.0,
        field1, from1_u, to1_u, count1_u,
        field2, from2_u, to2_u, count2_u,
        suppress1, suppress2,
    )

def import_warnings(metadata) -> list[str]:
    """Which of a restored file's checkpoint/style ids aren't in the current registry —
    for Import's own status message only. metadata_to_control_values() already falls
    back to a safe default for these regardless, so the app never crashes on an old
    export whose checkpoint/style has since been retired (e.g. a cut LoRA); this just
    lets the caller say so instead of silently substituting."""
    first = _cells(metadata)[0] if _cells(metadata) else {}
    warnings = []
    checkpoint_id = first.get("checkpoint_id")
    if checkpoint_id and checkpoint_id not in {cp.id for cp in CHECKPOINTS}:
        warnings.append(f"checkpoint '{checkpoint_id}'")
    lora_id = first.get("lora_id")
    if lora_id and lora_id not in {lora.id for lora in LORAS}:
        warnings.append(f"style '{lora_id}'")
    return warnings

def _slug(text: str, limit: int = 40) -> str:
    slug = re.sub(r"[^a-z0-9]+", "-", (text or "").lower()).strip("-")
    return slug[:limit].strip("-")

def _prepare_download_files(entry: dict) -> tuple[str, str]:
    """Builds (and caches on the entry) the PNG + settings-JSON files a download
    button hands out — cached so revisiting a gallery image doesn't rewrite files."""
    if entry.get("files"):
        return entry["files"]
    metadata = entry["metadata"]
    cells = _cells(metadata)
    stamp = time.strftime("%Y%m%d-%H%M%S", time.localtime(entry.get("created_at", time.time())))
    slug = _slug(cells[0].get("prompt", "") if cells else "")
    base = f"latent-studio_{stamp}_{slug}" if slug else f"latent-studio_{stamp}"

    folder = tempfile.mkdtemp(prefix="latent-studio-")
    png_path = os.path.join(folder, f"{base}.png")
    save_with_metadata(entry["image"], metadata or {}, png_path)
    json_path = os.path.join(folder, f"{base}_settings.json")
    with open(json_path, "w", encoding="utf-8") as handle:
        handle.write(settings_json(metadata))

    entry["files"] = (png_path, json_path)
    return entry["files"]

def history_to_gallery(history: list[dict]) -> list:
    # No captions — the settings live under the image and in the Advanced column.
    return [entry["image"] for entry in history]

def square_shape(count: int) -> tuple[int, int]:
    cols = math.ceil(math.sqrt(count))
    rows = math.ceil(count / cols)
    return rows, cols


### Event handlers + interactive logic
_(`src/latent_studio/callbacks.py`)_

The app's interactive brain: the `manager`/`controlnet_manager` pipeline singletons (module-global, since the app is single-user/local), the gating logic that greys out controls, and every `on_*` event handler. Two rules drive the greying: a setting the Compare tab is currently sweeping is locked to that tab, so it has only one source; and Style strength / Reference strength need a style / reference picked first. The Compare tab's own field radio is its on/off switch ("Nothing" = off, no separate checkbox), and picking `seed` there forces Seed mode to Fixed in the same `gating_updates()` call — a random base would silently change what a seed comparison even means. `on_generate` is the one dispatch point for both modes (single image / comparison grid) and both pipelines (plain / ControlNet): the actual diffusers call blocks, so it runs on a worker thread while the generator yields the polled `ProgressTracker` HTML a few times a second — that's what puts a real bar and ETA in the state panel for the model download, the denoising steps, and 'image 7 of 25' alike. The Generate button itself carries the live phase ('Loading model…' / 'Generating…' / 'Unloading model…') plus how many images a click produces ('Generate 5 images'); the state panel carries only the bars/ETA and any error in plain language, since repeating the phase there too would just be the same words twice — errors never become a popup, so there's never a duplicate message either. `on_import_settings` is the same idea from the other direction: a settings JSON, or a PNG's embedded metadata via `metadata.read_metadata`, applied to the live controls exactly like 'Reuse these settings' — the only difference is the source is a file, not the current result, so it works across sessions too.

In [ ]:
#@title callbacks.py
"""Event handlers and the app's interactive logic: which controls grey each other
out, and what happens on every button click / control change. Holds the app's two
pipeline singletons (single-user/local, so module-global is fine)."""

manager = PipelineManager()
controlnet_manager = ControlNetManager()

# checkpoint/lora/weight/prompt/negative/cfg/steps/seed_mode/seed/controlnet/
# controlnet_scale/field1/from1/to1/count1/field2/from2/to2/count2/suppress1/suppress2
APPLY_OUTPUT_COUNT = 21

def model_status_html() -> str:
    if manager.is_ready:
        return status_html("check", f"Loaded: {get_checkpoint(manager.checkpoint_id).label}")
    if controlnet_manager.is_ready:
        return status_html("check", f"Loaded: {get_checkpoint(controlnet_manager.checkpoint_id).label} + reference")
    return status_html("alert", "No model loaded yet — the first Generate loads one.")

def _image_suffix(count: int) -> str:
    return "image" if count <= 1 else f"{count} images"

def generate_button_label(
    checkpoint_id: str, lora_id: str, count: int = 1, controlnet_active: bool = False
) -> str:
    """The button's resting label: what a click will actually do, and how many images
    it will produce — folds in what used to be a separate "N images" note.
    controlnet_active picks which of the two pipeline singletons is actually going to
    serve the next click — a ControlNet generation loads/reads controlnet_manager, not
    manager (which work() unloads whenever ControlNet is on), so checking the wrong one
    here always reported "Load model & generate" even with a warm ControlNet pipeline."""
    suffix = _image_suffix(count)
    active = controlnet_manager if controlnet_active else manager
    if not active.is_ready or active.checkpoint_id != checkpoint_id:
        return f"Load model & generate {suffix}"
    if active.lora_id != lora_id:
        return f"Load style & generate {suffix}"
    return f"Generate {suffix}"

def _tracker_status(tracker: ProgressTracker):
    """Adapter: turns a manager's free-text on_status message into a tracker phase +
    coarse button state. "Unloading…" is the only sub-phase worth its own button text;
    everything else during a load reads as "Loading model…" on the button."""
    def on_status(message: str) -> None:
        button = "Unloading model…" if message.lower().startswith("unloading") else "Loading model…"
        tracker.set_phase(message, button)
    return on_status

# -- gating: which controls grey out, and why (shown in their info=) ----------

def compared_fields(field1: str, field2: str) -> set[str]:
    if field1 not in SWEEP_SPECS:  # field1 == "off" means Compare is off entirely
        return set()
    return {field for field in (field1, field2) if field in SWEEP_SPECS}

def gating_updates(field1, field2, lora_id, seed_mode, controlnet_select):
    """Updates for (cfg, steps, style strength, seed mode, seed, randomize, reference
    strength) — each control's info= carries only its live STATUS reason. Sweeping the
    seed also forces Fixed mode in the same update: a random base would silently change
    what a seed comparison even means."""
    compared = compared_fields(field1, field2)

    style_swept = "lora_weight" in compared
    style_off = lora_id == "none"
    style_info = STATUS_SWEPT if style_swept else (STATUS_STYLE_OFF if style_off else "")

    seed_swept = "seed" in compared
    seed_mode_now = "fixed" if seed_swept else seed_mode
    seed_info = STATUS_SWEPT if seed_swept else (STATUS_SEED_RANDOM if seed_mode_now == "random" else "")
    seed_mode_update = (
        gr.update(interactive=False, value="fixed") if seed_swept else gr.update(interactive=True)
    )

    return (
        gr.update(interactive="cfg_scale" not in compared, info=STATUS_SWEPT if "cfg_scale" in compared else ""),
        gr.update(interactive="steps" not in compared, info=STATUS_SWEPT if "steps" in compared else ""),
        gr.update(interactive=not style_swept and not style_off, info=style_info),
        seed_mode_update,
        gr.update(interactive=not seed_swept and seed_mode_now == "fixed", info=seed_info),
        gr.update(interactive=not seed_swept and seed_mode_now == "fixed"),
        gr.update(interactive=controlnet_select != "off", info="" if controlnet_select != "off" else STATUS_REFERENCE_OFF),
    )

def dim_updates(master_field: str, field: str, reset: bool, gate_field: bool = False, suppress: bool = False):
    """Updates for one Compare column: (field radio, start, end, steps, suppress-flag
    cleared back to False). The first column's radio is the on/off switch itself, so it
    always stays clickable (gate_field=False, the default) — but the second column only
    makes sense once the first is active, so its own radio greys out too when
    gate_field=True. Start/End grey out when off or when the field is seed (seed has no
    numeric range, only a count); `reset` re-ranges Start/End/Steps to the new field's
    default (range and count both).

    `suppress=True` means a restore (Reuse/Import) just set this column's exact values
    — Start/End/Steps alike — as part of the very same event that changed `field`; this
    cascade would otherwise fire right after and immediately overwrite them with the
    field's generic default. Skip the *value* reset for that one cycle and clear the flag
    so the next real field change resets as usual — but still realign every control's
    minimum/maximum(/step) to the restored field's own spec (never suppressed): each of
    Start/End/Steps carries its range over from whatever field was active before, and the
    four fields' ranges genuinely don't nest (lora_weight's floor of 0.0 sits below every
    other field's minimum; cfg_scale/steps/lora_weight cap Steps at 8 images where seed
    goes to 25). A restored value landing outside the previous field's leftover range
    crashes Gradio's own bounds check in preprocess() before on_generate ever runs — only
    realigning the range here (not widening it to some shared safe superset) fixes that
    without losing what each field's own range means, both what the Start/End sliders
    visibly drag across and Steps' per-field image-count ceiling."""
    compare_on = master_field in SWEEP_SPECS
    spec = SWEEP_SPECS.get(field)
    usable = compare_on and spec is not None
    is_seed = field == "seed"
    numeric_on = usable and not is_seed

    if suppress:
        field_update = gr.update(interactive=compare_on) if gate_field else gr.update()
        if usable:
            from_update = gr.update(
                interactive=numeric_on, minimum=spec.minimum, maximum=spec.maximum, step=spec.step
            )
            to_update = gr.update(
                interactive=numeric_on, minimum=spec.minimum, maximum=spec.maximum, step=spec.step
            )
            count_update = gr.update(interactive=usable, maximum=spec.max_count)
        else:
            from_update = gr.update(interactive=numeric_on)
            to_update = gr.update(interactive=numeric_on)
            count_update = gr.update(interactive=usable)
        return (field_update, from_update, to_update, count_update, False)

    if numeric_on and reset:
        from_update = gr.update(
            interactive=True, minimum=spec.minimum, maximum=spec.maximum, step=spec.step, value=spec.start
        )
        to_update = gr.update(
            interactive=True, minimum=spec.minimum, maximum=spec.maximum, step=spec.step, value=spec.end
        )
    else:
        from_update = gr.update(interactive=numeric_on)
        to_update = gr.update(interactive=numeric_on)

    count_update = (
        gr.update(interactive=True, maximum=spec.max_count, value=spec.count)
        if usable and reset
        else gr.update(interactive=usable)
    )
    field_update = gr.update(interactive=compare_on) if gate_field else gr.update()
    return field_update, from_update, to_update, count_update, False

def image_count(field1, count1, field2, count2) -> int:
    if field1 not in SWEEP_SPECS:
        return 1
    total = int(count1)
    if field2 in SWEEP_SPECS:
        total *= int(count2)
    return total

# -- event handlers -------------------------------------------------------------

def on_button_label_change(checkpoint_id, lora_id, field1, count1, field2, count2):
    count = image_count(field1, count1, field2, count2)
    return gr.update(value=generate_button_label(checkpoint_id, lora_id, count))

def on_randomize_seed():
    return random.randint(0, MAX_SEED)

def on_advanced_toggle(enabled: bool):
    return gr.update(visible=enabled)

def on_controlnet_toggle(enabled: bool):
    """Reference image is Advanced-only — it rides on the same toggle as the Settings
    panel and shows/hides alongside it. Turning Advanced off also resets the
    reference-type radio to "off" rather than leaving it selected underneath —
    turning Advanced back on later should mean picking a reference type again, not
    silently reactivating whatever was chosen before it was hidden."""
    return gr.update(visible=enabled), gr.update() if enabled else "off"

def on_controlnet_change(controlnet_select: str):
    # The upload + Process button grey out instead of the row disappearing while no
    # reference type is picked — matches the app's never-hide-a-control rule. Also
    # clears the stale preview so switching outlines<->depth re-runs the preprocessor.
    active = controlnet_select != "off"
    return gr.update(interactive=active), gr.update(interactive=active), None

def on_preprocess(image, controlnet_select: str):
    if image is None:
        raise gr.Error("Upload a reference image first.")
    if controlnet_select == "canny":
        return canny_preprocess(image)
    if controlnet_select == "depth":
        return depth_preprocess(image)
    raise gr.Error("Pick a reference type first.")

def on_preload():
    """Warms the disk cache with every checkpoint + LoRA (no VRAM used), on the same
    worker-thread + ProgressTracker machinery as Generate, so a live demo's first
    model switch isn't a download wait."""
    tracker = ProgressTracker()
    outcome: dict = {}

    def work():
        try:
            with track_tqdm(tracker):
                preload_models(lambda msg: tracker.set_phase(msg, "Preloading…"))
        except Exception as exc:  # noqa: BLE001 — surfaced in the panel, never a popup
            outcome["error"] = exc

    worker = threading.Thread(target=work, daemon=True)
    worker.start()
    while worker.is_alive():
        yield gr.update(interactive=False, value="Preloading…"), tracker.html()
        time.sleep(0.2)
    worker.join()

    if "error" in outcome:
        yield gr.update(interactive=True, value="Preload all models"), status_html("alert", str(outcome["error"]))
        return
    yield (
        gr.update(interactive=True, value="Preload all models"),
        status_html("check", "All models cached — the first switch will be instant."),
    )

def on_import_settings(file):
    """Import: apply a settings JSON, or a PNG's embedded metadata, to the live
    controls — exactly like "Reuse these settings", but from a file, so it works
    across sessions too. Errors are plain language in the state panel, never a popup.

    A generator, not a plain return: a large grid export's embedded metadata is a
    real parse (and the upload itself can take a moment), and the fields it's about
    to overwrite — checkpoint, style, Compare's own sweep config — are exactly what
    Generate reads. The interim "Importing…" yield disables Generate for that window
    so a click mid-import can't race the restore and fire on a half-applied state."""
    noop = tuple(gr.update() for _ in range(APPLY_OUTPUT_COUNT))
    if file is None:
        yield (*noop, status_html("alert", "No file selected."), gr.update())
        return

    yield (*noop, status_html("upload", "Importing settings…"), gr.update(interactive=False, value="Importing…"))

    path = file if isinstance(file, str) else file.name
    ext = os.path.splitext(path)[1].lower()
    try:
        if ext == ".json":
            with open(path, encoding="utf-8") as handle:
                metadata = json.load(handle)
        else:
            metadata = read_metadata(path)
            if metadata is None:
                yield (*noop, status_html(
                    "alert",
                    "This image has no Latent Studio metadata embedded — was it downloaded from this app?",
                ), gr.update(interactive=True))
                return
    except json.JSONDecodeError as exc:
        yield (*noop, status_html("alert", f"Not a valid settings file: {exc}"), gr.update(interactive=True))
        return
    except Exception as exc:  # noqa: BLE001 — surfaced in the panel, never a popup
        yield (*noop, status_html("alert", f"Couldn't read that file: {exc}"), gr.update(interactive=True))
        return

    values = metadata_to_control_values(metadata)
    checkpoint_id, lora_id, field1, field2 = values[0], values[1], values[11], values[15]
    count1, count2 = values[14].get("value", 1), values[18].get("value", 1)
    total = image_count(field1, count1, field2, count2)

    warnings = import_warnings(metadata)
    message = (
        f"Settings imported, but {' and '.join(warnings)} no longer exist — defaulted instead."
        if warnings
        else "Settings imported — press Generate to reproduce it."
    )
    yield (
        *values,
        status_html("alert" if warnings else "check", message),
        gr.update(interactive=True, value=generate_button_label(checkpoint_id, lora_id, total)),
    )

def on_generate(
    checkpoint_id,
    lora_id,
    prompt,
    negative_prompt,
    cfg_scale,
    steps,
    seed_mode,
    seed,
    lora_weight,
    advanced_view_on,
    controlnet_select,
    controlnet_preview,
    controlnet_scale,
    field1, from1, to1, count1,
    field2, from2, to2, count2,
    history,
):
    """The one dispatch point for both modes (single image / comparison grid) and
    both pipelines (plain / ControlNet). Runs the diffusers call on a worker thread
    and polls a ProgressTracker so the state panel can show a real bar + ETA."""
    compare_enabled = field1 in SWEEP_SPECS
    total_images = image_count(field1, count1, field2, count2)

    def label():
        return generate_button_label(checkpoint_id, lora_id, total_images, controlnet_active=bool(controlnet_types))

    def frozen(state_html):
        return (
            gr.update(), history, gr.update(), gr.update(), gr.update(), gr.update(),
            gr.update(interactive=True, value=label()),
            model_status_html(), state_html, gr.update(), gr.update(), gr.update(),
        )

    def failed(message):
        return frozen(status_html("alert", message))

    # ControlNet is experimental and Advanced-only — only build a real controlnet_types
    # list when Advanced view is actually on, regardless of what's selected underneath
    # (matches on_controlnet_toggle resetting controlnet_select to "off" whenever
    # Advanced itself goes off).
    controlnet_types = [controlnet_select] if advanced_view_on and controlnet_select != "off" else []
    control_images: dict = {}
    controlnet_scales: dict = {}
    if controlnet_types:
        if controlnet_preview is None:
            yield failed("Process the reference image first — upload one and press Process.")
            return
        control_images = {controlnet_select: controlnet_preview}
        controlnet_scales = {controlnet_select: controlnet_scale}

    if compare_enabled:
        if field2 in SWEEP_SPECS and field1 == field2:
            yield failed("Pick two different settings to compare.")
            return
        if total_images > MAX_IMAGES:
            yield failed(f"That's {total_images} images. Keep a comparison under {MAX_IMAGES}.")
            return

    # Random mode rolls the seed once, here, and reports it back into the (still
    # greyed) seed box — so a lucky result can be pinned by switching to Fixed.
    used_seed = random.randint(0, MAX_SEED) if seed_mode == "random" else int(seed)

    def dim(field, from_value, to_value, count):
        if field == "seed":
            # Compare-mode seeds always count up by 1 from the Tuning tab's own seed —
            # gating_updates() already forces Fixed mode whenever a seed sweep is picked.
            return ("seed", (used_seed, 1), int(count))
        return (field, (float(from_value), float(to_value)), int(count))

    base_params = GenerationParams(
        prompt=prompt.strip(),
        negative_prompt=negative_prompt.strip(),
        width=WIDTH,
        height=HEIGHT,
        cfg_scale=cfg_scale,
        steps=int(steps),
        seed=used_seed,
        lora_weight=lora_weight,
        controlnet_types=controlnet_types,
        controlnet_scales=controlnet_scales,
    )

    tracker = ProgressTracker()
    outcome: dict = {}

    def work():
        try:
            with track_tqdm(tracker):
                if controlnet_types:
                    if manager.is_ready:
                        manager.unload()  # only one full pipeline stays resident
                    stale = not (
                        controlnet_manager.is_ready
                        and controlnet_manager.checkpoint_id == checkpoint_id
                        and controlnet_manager.active_types == controlnet_types
                    )
                    if stale:
                        tracker.set_phase("Loading the reference model…", "Loading model…")
                        controlnet_manager.load(checkpoint_id, controlnet_types)
                    if controlnet_manager.lora_id != lora_id:
                        tracker.set_phase(f"Loading the {get_lora(lora_id).label} style…", "Loading model…")
                        controlnet_manager.set_lora(lora_id)
                else:
                    if controlnet_manager.is_ready:
                        controlnet_manager.unload()
                    if not manager.is_ready or manager.checkpoint_id != checkpoint_id:
                        checkpoint_label = get_checkpoint(checkpoint_id).label
                        tracker.set_phase(
                            f"Loading {checkpoint_label} — downloaded once, then cached", "Loading model…"
                        )
                        manager.load_checkpoint(checkpoint_id, on_status=_tracker_status(tracker))
                    if manager.lora_id != lora_id:
                        tracker.set_phase(f"Loading the {get_lora(lora_id).label} style…", "Loading model…")
                        manager.set_lora(lora_id, on_status=_tracker_status(tracker))

                tracker.set_phase("Generating…", "Generating…", image_total=total_images)

                def generate_fn(params):
                    return generate(manager, controlnet_manager, params, control_images)

                if not compare_enabled:
                    image, metadata = generate_fn(base_params)
                else:
                    dim1 = dim(field1, from1, to1, count1)
                    dim2 = dim(field2, from2, to2, count2) if field2 in SWEEP_SPECS else None
                    # Seeds have no order, so a row says less than a block; a swept
                    # number does have an order and reads best left to right.
                    shape = square_shape(int(count1)) if dim2 is None and field1 == "seed" else None
                    image, metadata = sweep(
                        generate_fn, base_params, dim1, dim2, shape=shape, progress=tracker.on_image
                    )
                    # Store the exact sweep config alongside the cells — Import/Reuse
                    # restore it verbatim (metadata_view.metadata_to_control_values),
                    # rather than trying to reverse-engineer which field was which axis
                    # from the cells' bare values (ambiguous, and doesn't work at all
                    # for "Reuse" on a grid either).
                    metadata = {
                        "cells": metadata,
                        "compare": {
                            "field1": field1, "from1": float(from1), "to1": float(to1), "count1": int(count1),
                            "field2": field2 if field2 in SWEEP_SPECS else "off",
                            "from2": float(from2) if field2 in SWEEP_SPECS else None,
                            "to2": float(to2) if field2 in SWEEP_SPECS else None,
                            "count2": int(count2) if field2 in SWEEP_SPECS else None,
                        },
                    }
                outcome["image"], outcome["metadata"] = image, metadata
        except Exception as exc:  # noqa: BLE001 — every failure belongs in the panel
            outcome["error"] = exc

    worker = threading.Thread(target=work, daemon=True)
    worker.start()
    while worker.is_alive():
        yield (
            gr.update(), history, gr.update(), gr.update(), gr.update(), gr.update(),
            gr.update(interactive=False, value=tracker.button),
            model_status_html(), tracker.html(), gr.update(), gr.update(), gr.update(),
        )
        time.sleep(0.2)
    worker.join()

    if "error" in outcome:
        yield failed(str(outcome["error"]))
        return

    image, metadata = outcome["image"], outcome["metadata"]
    entry = {"image": image, "metadata": metadata, "created_at": time.time()}
    history = [entry] + history
    png_path, json_path = _prepare_download_files(entry)

    cells = _cells(metadata)
    done = f"Done — {len(cells)} images" if len(cells) > 1 else f"Done — seed {cells[0].get('seed', '?')}"

    yield (
        metadata, history,
        gr.update(value=history_to_gallery(history), selected_index=0), 0,
        gr.update(value=png_path, visible=True), json_path,
        gr.update(interactive=True, value=label()),
        model_status_html(), status_html("check", done),
        prompt_used_html(metadata), settings_html(metadata),
        gr.update(value=used_seed),
    )

def on_gallery_select(evt: gr.SelectData, history):
    # Shows the selected thumbnail's prompt/settings and re-points the download
    # buttons — deliberately doesn't touch the live controls ("Reuse" does that).
    entry = history[evt.index]
    png_path, json_path = _prepare_download_files(entry)
    metadata = entry["metadata"]
    return (
        metadata, evt.index, gr.update(value=png_path, visible=True), json_path,
        prompt_used_html(metadata), settings_html(metadata),
    )


### App layout
_(`src/latent_studio/app.py`)_

`build_app()` assembles the `gr.Blocks` UI and wires every control to its handler from callbacks.py — left column: prompt, avoid, the Generate button, the live state panel; center: the image and the prompt that made it; right (Advanced only): the exact settings, 'Reuse these settings', Import/Export (a settings JSON or a PNG this app produced, applied the same way Reuse does), and the image download button. Everything else sits in four tabs (Model & style, Tuning, Reference image, Compare).

Two rules shape the layout. **Nothing containing a slider is ever hidden** — Gradio mounts a slider inside a `display:none` container at zero width and it stays invisible until touched, so a control that doesn't apply right now is greyed out instead, with the reason in its info line (`callbacks.gating_updates`). And **a setting has exactly one source** — a setting the Compare tab is sweeping greys out its normal control and says so. The `_title`/`_description`/`_heading`/`_compare_column` helpers exist because every control goes `container=False` (dropping Gradio's own component label), so each control's title/description is a plain Markdown line above it instead — deliberately not wrapped in `gr.Group()`, whose own box comes from a Gradio CSS quirk (`--border-color-primary`, not `--block-background-fill`) with no clean theme lever to switch off, since that variable is also the real border color for unrelated elements elsewhere (gallery thumbnails, focus rings). The last cell calls `build_app().launch(theme=THEME)` — Gradio 6 takes `theme` on `launch()`, not the `Blocks` constructor.

In [ ]:
#@title app.py
"""Gradio app layout: build_app() assembles the gr.Blocks UI and wires every
control to its handler in callbacks.py. One Generate button drives both modes
(single image / comparison grid) and both pipelines.

Layout rule: nothing containing a slider is ever hidden — Gradio mounts a
slider inside a zero-width display:none container and it stays invisible
until touched — so controls that don't apply right now are greyed out
instead, with the reason in their info line (see callbacks.gating_updates)."""

def _title(text: str) -> gr.Markdown:
    """A control's title, as an h3 — container=False drops the normal component label."""
    return gr.Markdown(f"### {text}")

def _description(text: str) -> gr.Markdown:
    return gr.Markdown(text)

def _sublabel(text: str) -> gr.Markdown:
    """A small bold label for a sub-field (Compare tab's Start/End/Steps)."""
    return gr.Markdown(f"**{text}**")

def _heading(title: str, description: str) -> None:
    """Title + description pair shown above a control. Deliberately not wrapped in a
    gr.Group() — Group's own background/border turned out to come from a Gradio CSS
    quirk (--border-color-primary, shared with unrelated elements) with no clean theme
    lever to switch off, so plain spacing wins over a custom CSS chase."""
    _title(title)
    _description(description)

def _compare_column(heading: str, description: str, choices: list, value: str, spec, interactive: bool = True):
    """One column of the Compare tab. The first column's field radio is its own on/off
    switch ("Nothing" = off) and stays clickable; the second column's only makes sense
    once the first is active, so it starts (and re-)greys via `interactive`/
    dim_updates()'s gate_field. Start/End/Steps grey out until a real setting is picked,
    and dim_updates() re-ranges them for whichever setting that is."""
    _heading(heading, description)
    field = gr.Radio(choices=choices, value=value, container=False, info="", interactive=interactive)
    low, high, step = (spec.minimum, spec.maximum, spec.step) if spec else (1, 100, 1)
    start, end = (spec.start, spec.end) if spec else (10, 50)
    with gr.Row():
        with gr.Column(min_width=140):
            _sublabel("Start")
            from_slider = gr.Slider(low, high, value=start, step=step, container=False, interactive=False)
        with gr.Column(min_width=140):
            _sublabel("End")
            to_slider = gr.Slider(low, high, value=end, step=step, container=False, interactive=False)
        with gr.Column(min_width=100):
            _sublabel("Steps")
            count = gr.Number(
                value=spec.count if spec else 5, precision=0, minimum=2,
                maximum=spec.max_count if spec else 10, container=False, interactive=False,
            )
    return field, from_slider, to_slider, count

def build_app() -> gr.Blocks:
    checkpoint_choices = [(cp.label, cp.id) for cp in CHECKPOINTS]
    lora_choices = [(lora.label, lora.id) for lora in LORAS]
    compare_choices = [("Nothing", "off")] + SWEEP_CHOICES

    with gr.Blocks(title="Latent Studio") as demo:
        history_state = gr.State([])
        metadata_state = gr.State(None)  # the selected image's metadata; drives Reuse
        selected_index_state = gr.State(0)
        # Set True by Reuse/Import alongside a restored field1/field2 to tell that
        # column's own dim_updates() cascade to skip its usual value-reset just once —
        # see dim_updates()'s docstring.
        suppress1_state = gr.State(False)
        suppress2_state = gr.State(False)

        gr.Markdown(
            "# Latent Studio\n"
            "_Describe a scene, then let an artist paint it._ Six hand-trained styles — "
            "from Hokusai's waves to Rembrandt's candlelight — are ready to try alongside "
            "your own prompts, with every setting behind them yours to tune and reproduce. "
            "Press **Generate** to start; the first run downloads a model (a few minutes), "
            "every one after takes seconds."
        )

        # ---------------- main area ----------------
        with gr.Row():
            # LEFT — prompt, button, live state. Each control is a title + description +
            # field in sequence; Gradio's own layout gap gives them breathing room.
            with gr.Column(scale=3, variant="panel"):
                _heading("Prompt", DESC_PROMPT)
                prompt_input = gr.Textbox(
                    container=False,
                    value=random.choice(EXAMPLE_PROMPTS),
                    placeholder="Describe the image you want…",
                    lines=3,
                )
                _heading("Avoid", DESC_AVOID)
                negative_prompt_input = gr.Textbox(
                    container=False,
                    placeholder="e.g. blurry, low quality, text…",
                    lines=2,
                )
                generate_button = gr.Button(
                    generate_button_label(DEFAULT_CHECKPOINT_ID, DEFAULT_LORA_ID),
                    variant="primary",
                    size="lg",
                    icon=button_icon("sparkles", ICON_ON_PRIMARY),
                )
                state_info = gr.HTML(
                    status_html("check", "Ready."), container=False, padding=False,
                    apply_default_css=False, elem_id="state_info",
                )

            # CENTER — the picture and the prompt that made it.
            with gr.Column(scale=6):
                # Gradio's preview image is exactly gallery height minus a fixed 60px
                # thumbnail strip (measured in the shipped component CSS) — 572 renders
                # the 512px image at its true resolution instead of shrunk to ~500px.
                history_gallery = gr.Gallery(
                    show_label=False, container=False, preview=True, selected_index=0,
                    columns=8, height=572, object_fit="contain", buttons=[],
                )
                prompt_used = gr.HTML(
                    prompt_used_html(None), container=False, padding=False, apply_default_css=False
                )

            # RIGHT — Advanced only: exact settings, the way back to them, and the way to
            # keep the image itself.
            with gr.Column(scale=3, visible=False) as advanced_column:
                _title("Settings")
                settings_view = gr.HTML(
                    settings_html(None), container=False, padding=False, apply_default_css=False
                )
                apply_settings_button = gr.Button(
                    "Reuse these settings", size="sm", icon=button_icon("reuse")
                )
                with gr.Row():
                    # Uploads a settings JSON or a PNG this app produced (its embedded
                    # metadata is read back out) and applies it like "Reuse" does —
                    # copying the box above by hand already covers the old Copy button.
                    import_button = gr.UploadButton(
                        "Import", size="sm", icon=button_icon("upload"), file_types=[".json", ".png"],
                    )
                    export_button = gr.DownloadButton("Export", size="sm", icon=button_icon("download"))
                download_image_button = gr.DownloadButton(
                    "Download image", size="sm", icon=button_icon("download"), visible=False
                )

        # ---------------- settings ----------------
        with gr.Tabs():
            with gr.Tab("Model & style"):
                with gr.Row():
                    with gr.Column():
                        _heading("Model", DESC_MODEL)
                        checkpoint_radio = gr.Radio(
                            choices=checkpoint_choices, value=DEFAULT_CHECKPOINT_ID,
                            container=False, info="",
                        )
                        model_status = gr.HTML(
                            model_status_html(), container=False, padding=False, apply_default_css=False
                        )
                    with gr.Column():
                        _heading("Style", DESC_STYLE)
                        lora_radio = gr.Radio(
                            choices=lora_choices, value=DEFAULT_LORA_ID, container=False, info="",
                        )
                        _heading("Style strength", DESC_STYLE_STRENGTH)
                        lora_weight_slider = gr.Slider(
                            minimum=0.0, maximum=1.5, step=0.05, value=1.0, container=False,
                            info=STATUS_STYLE_OFF, interactive=False,
                        )

            with gr.Tab("Tuning"):
                with gr.Row():
                    with gr.Column():
                        _heading("Prompt strength", DESC_CFG)
                        cfg_slider = gr.Slider(
                            minimum=1.0, maximum=20.0, step=0.05, value=7.5, container=False, info="",
                        )
                        _heading("Detail", DESC_STEPS)
                        steps_slider = gr.Slider(
                            minimum=1, maximum=100, step=1, value=30, container=False, info="",
                        )
                    with gr.Column():
                        _heading("Seed", DESC_SEED)
                        seed_mode = gr.Radio(
                            choices=[("A new one every time", "random"), ("Always the same", "fixed")],
                            value="random", container=False, info="",
                        )
                        seed_input = gr.Number(
                            value=1312, precision=0, container=False,
                            info=STATUS_SEED_RANDOM, interactive=False,
                        )
                        randomize_button = gr.Button(
                            "Roll a new seed", size="sm", icon=button_icon("dice"), interactive=False
                        )

            # ControlNet (ADV) is experimental — Advanced view (Setup tab) is the only way
            # in; the tab itself stays hidden until Advanced is on, and on_generate only
            # builds a real controlnet_types list in that case. The core generator works
            # fully with this off, which is the default (Advanced itself defaults off).
            with gr.Tab("Reference image", visible=False) as reference_tab:
                with gr.Row():
                    with gr.Column():
                        _heading("Copy a shape from an image", DESC_REFERENCE)
                        controlnet_select = gr.Radio(
                            choices=CONTROLNET_CHOICES, value="off", container=False, info="",
                        )
                    with gr.Column():
                        _heading("Reference strength", DESC_REFERENCE_SCALE)
                        controlnet_scale = gr.Slider(
                            0.0, 2.0, value=1.0, step=0.05, container=False,
                            info=STATUS_REFERENCE_OFF, interactive=False,
                        )
                with gr.Row():
                    with gr.Column():
                        _sublabel("Your reference")
                        controlnet_input_image = gr.Image(
                            container=False, type="pil", sources=["upload"], height=300, interactive=False,
                        )
                        controlnet_preprocess_button = gr.Button(
                            "Process", size="sm", interactive=False
                        )
                    with gr.Column():
                        _sublabel("What the model will actually follow")
                        controlnet_preview = gr.Image(
                            container=False, type="pil", interactive=False, height=300,
                        )
                gr.Markdown(
                    "_Your chosen style still applies — the reference fixes the composition, "
                    "the style paints it._"
                )

            with gr.Tab("Compare"):
                _heading("Compare settings", DESC_COMPARE)
                with gr.Row():
                    with gr.Column():
                        field1, from1, to1, count1 = _compare_column(
                            "Compare this",
                            'The setting to walk across the grid, left to right. Leave it on '
                            '"Nothing" for a single image.',
                            compare_choices, "off", None,
                        )
                    with gr.Column():
                        field2, from2, to2, count2 = _compare_column(
                            "And this",
                            "Optional — add a second setting to lay it out top to bottom as well, "
                            "turning the row into a full table.",
                            compare_choices, "off", None, interactive=False,
                        )

            with gr.Tab("Setup"):
                with gr.Row():
                    with gr.Column():
                        _heading("Advanced view", DESC_ADVANCED)
                        advanced_toggle = gr.Checkbox(
                            label="Show each image's exact settings",
                            value=False, container=False, info="",
                        )
                    with gr.Column():
                        _heading("Preload models", DESC_PRELOAD)
                        preload_button = gr.Button(
                            "Preload all models", size="sm", icon=button_icon("download")
                        )

        # ---------------- wiring ----------------
        gating_inputs = [field1, field2, lora_radio, seed_mode, controlnet_select]
        gating_outputs = [
            cfg_slider, steps_slider, lora_weight_slider,
            seed_mode, seed_input, randomize_button, controlnet_scale,
        ]
        label_inputs = [checkpoint_radio, lora_radio, field1, count1, field2, count2]
        dim1_outputs = [field1, from1, to1, count1, suppress1_state]
        dim2_outputs = [field2, from2, to2, count2, suppress2_state]
        # Reuse/Import restore field1/field2/Start/End/Steps from the grid's own stored
        # sweep config (or reset Compare to off for a single image / an older export
        # with no stored config) — see metadata_view.metadata_to_control_values. The
        # suppress flags ride along so the field-radio's own re-ranging cascade doesn't
        # immediately clobber the values just restored in this same update.
        apply_outputs = [
            checkpoint_radio, lora_radio, lora_weight_slider, prompt_input, negative_prompt_input,
            cfg_slider, steps_slider, seed_mode, seed_input, controlnet_select, controlnet_scale,
            field1, from1, to1, count1, field2, from2, to2, count2,
            suppress1_state, suppress2_state,
        ]

        # Every control that can grey another one out recomputes the whole gating set.
        # show_progress="hidden" throughout this block: these are instant UI-state
        # recomputations, not loading work, so the components they touch shouldn't
        # flash into Gradio's default pending overlay.
        for trigger in (field1, field2, lora_radio, seed_mode, controlnet_select):
            trigger.change(gating_updates, inputs=gating_inputs, outputs=gating_outputs, show_progress="hidden")

        # The button's label carries both what a click will do (load model/style?) and
        # how many images it produces — recomputed by anything that affects either.
        for trigger in (checkpoint_radio, lora_radio, field1, field2, count1, count2):
            trigger.change(
                on_button_label_change, inputs=label_inputs, outputs=generate_button, show_progress="hidden"
            )

        # field1 is Compare's own on/off switch now — picking a setting re-ranges its
        # own Start/End/Steps. field2 only re-ranges on its own change; a field1 change
        # just re-evaluates whether field2's column is usable, without resetting it.
        field1.change(
            lambda f, s: dim_updates(f, f, reset=True, suppress=s),
            inputs=[field1, suppress1_state], outputs=dim1_outputs,
            show_progress="hidden",
        )
        field1.change(
            lambda f1, f2, s: dim_updates(f1, f2, reset=False, gate_field=True, suppress=s),
            inputs=[field1, field2, suppress2_state], outputs=dim2_outputs,
            show_progress="hidden",
        )
        field2.change(
            lambda f1, f2, s: dim_updates(f1, f2, reset=True, gate_field=True, suppress=s),
            inputs=[field1, field2, suppress2_state], outputs=dim2_outputs,
            show_progress="hidden",
        )

        advanced_toggle.change(
            on_advanced_toggle, inputs=advanced_toggle, outputs=advanced_column, show_progress="hidden"
        )
        # Reference image rides on the same switch as the Settings panel — see
        # on_controlnet_toggle's docstring.
        advanced_toggle.change(
            on_controlnet_toggle, inputs=advanced_toggle, outputs=[reference_tab, controlnet_select],
            show_progress="hidden",
        )
        preload_button.click(
            on_preload, outputs=[preload_button, state_info], show_progress="hidden"
        )  # the button + state panel draw their own progress
        randomize_button.click(on_randomize_seed, outputs=seed_input, show_progress="hidden")
        controlnet_select.change(
            on_controlnet_change, inputs=controlnet_select,
            outputs=[controlnet_input_image, controlnet_preprocess_button, controlnet_preview],
            show_progress="hidden",
        )
        controlnet_preprocess_button.click(
            on_preprocess, inputs=[controlnet_input_image, controlnet_select], outputs=controlnet_preview,
            show_progress="hidden",
        )

        generate_button.click(
            on_generate,
            inputs=[
                checkpoint_radio, lora_radio, prompt_input, negative_prompt_input,
                cfg_slider, steps_slider, seed_mode, seed_input, lora_weight_slider,
                advanced_toggle, controlnet_select, controlnet_preview, controlnet_scale,
                field1, from1, to1, count1,
                field2, from2, to2, count2,
                history_state,
            ],
            outputs=[
                metadata_state, history_state, history_gallery, selected_index_state,
                download_image_button, export_button,
                generate_button, model_status, state_info,
                prompt_used, settings_view, seed_input,
            ],
            show_progress="hidden",  # the state panel draws its own bars
        )
        history_gallery.select(
            on_gallery_select, inputs=history_state,
            outputs=[
                metadata_state, selected_index_state, download_image_button, export_button,
                prompt_used, settings_view,
            ],
            show_progress="hidden",
        )
        apply_settings_button.click(
            metadata_to_control_values, inputs=metadata_state, outputs=apply_outputs, show_progress="hidden"
        )
        import_button.upload(
            on_import_settings, inputs=import_button, outputs=apply_outputs + [state_info, generate_button],
            show_progress="hidden",
        )

    return demo


## 5. Download the models (optional — off by default)
The app already downloads each model itself the first time you select it, with its own loading animation, and has a **"Preload all models"** button in its Setup tab for the same one-shot warm-up this cell does. Leaving `RUN_PREFLIGHT` at `False` skips this cell and relies on that — faster to reach the UI, and no cache space spent on a checkpoint or style this session never uses.

Turn it on if a Generate click hits repeated download errors. Since **2026-07-13** the Hub's CDN has intermittently rejected its own signed download links (`403 … SignatureError: invalid key pair id`, [huggingface/datasets#8328](https://github.com/huggingface/datasets/issues/8328)) — a Hugging Face outage, unrelated to this project, that hits public repos too and is still open as of the last check. This cell's retry loop pulls every checkpoint and style **now** and works its way through the flakiness: the cache **resumes**, so each attempt keeps whatever it already downloaded even if most of them fail. If a run dies anyway, just run the cell again — nothing is lost.

The styles are copied into `app_loras/`, which the pipeline prefers over the Hub, so after this cell they load with no network at all.

In [ ]:
#@title Download the models (optional)
RUN_PREFLIGHT = False  # @param {type:"boolean"}

if RUN_PREFLIGHT:
    import time

    RETRIES = 40


    def fetch(label, call):
        """Retry past the Hub CDN's intermittent 403s. The cache resumes, so every
        attempt makes progress even if most of them fail."""
        for attempt in range(1, RETRIES + 1):
            try:
                return call()
            except Exception as exc:
                flaky = any(s in str(exc) for s in ("403", "SignatureError", "Connection"))
                if not flaky or attempt == RETRIES:
                    raise
                print(f"  {label}: Hub CDN hiccup ({attempt}/{RETRIES}), retrying...")
                time.sleep(3)

    import os
    import shutil

    from diffusers import DiffusionPipeline
    from huggingface_hub import hf_hub_download


    def fetch_checkpoint(cp):
        # fp16 halves the download; not every community checkpoint publishes that
        # variant (epiCRealism doesn't), so fall back rather than fail.
        try:
            return fetch(
                cp.label,
                lambda: DiffusionPipeline.download(cp.repo_id, variant="fp16", use_safetensors=True),
            )
        except Exception:
            print(f"  {cp.label}: no fp16 build, fetching full weights...")
            return fetch(cp.label, lambda: DiffusionPipeline.download(cp.repo_id, use_safetensors=True))


    for cp in CHECKPOINTS:
        print(f"Fetching {cp.label}...")
        fetch_checkpoint(cp)

    os.makedirs(APP_LORAS_DIR, exist_ok=True)
    for lora in LORAS:
        if lora.path is None:
            continue
        print(f"Fetching {lora.label}...")
        weights = fetch(
            lora.label,
            lambda lora=lora: hf_hub_download(lora.path, "pytorch_lora_weights.safetensors"),
        )
        # Staged locally, so the app loads styles without touching the Hub again.
        shutil.copy(weights, os.path.join(APP_LORAS_DIR, f"{lora.id}-lora.safetensors"))

    print("\nAll models cached. The app will load them from disk.")
else:
    print(
        "Skipped — the app downloads each model itself on first use (with its own "
        "loading animation), or use its 'Preload all models' button. Tick "
        "RUN_PREFLIGHT above and re-run this cell if a Generate click hits repeated "
        "download errors."
    )

## 6. Launch the app
Switch to a GPU runtime first. `share=True` creates a public link outside this Colab session — the way to reach the app from another device, e.g. for a live demo. Load the printed link once to confirm it works before depending on it. `inbrowser=True` only opens a tab automatically on a **local** runtime — on Google's hosted runtime (the normal case for the T4 GPU) there's no local browser for it to reach, so use the printed share link instead; it's a harmless no-op either way, not an error.

In [ ]:
demo = build_app()
demo.launch(share=True, theme=THEME, inline=False, inbrowser=True, footer_links=[])